In [1]:
# Cell 01 — 패키지 설치
# ※ olefile, pdfplumber 제외 (파싱 불필요)
import sys
!{sys.executable} -m pip install chromadb openai rapidfuzz rank_bm25 \
    "sentence-transformers>=3.0.0" FlagEmbedding \
    ragas datasets \
    langchain-ollama langchain-huggingface \
    einops openpyxl -q

print("✅ Cell 01 완료 — 패키지 설치 완료")
print("  ※ EXAONE-3.5 미설치 시: ollama pull exaone3.5:7.8b")
print("  ※ phi4-mini 미설치 시  : ollama pull phi4-mini")

✅ Cell 01 완료 — 패키지 설치 완료
  ※ EXAONE-3.5 미설치 시: ollama pull exaone3.5:7.8b
  ※ phi4-mini 미설치 시  : ollama pull phi4-mini


In [2]:
# Cell 02 — 라이브러리 임포트 및 경로 설정

import os
import re
import json
import math
import time
import copy
import random
import warnings
import unicodedata
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

from openai import OpenAI
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from rapidfuzz import process, fuzz
from IPython.display import display, HTML
import html as html_lib

# ── 경로 설정 ─────────────────────────────────────────────────────
BASE_PATH  = Path("/Users/who/Desktop/code_it/project01/final_files")
DATA_PATH  = BASE_PATH / "data"
RESULT_DIR = BASE_PATH / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ── chunks 파일 탐색 ──────────────────────────────────────────────
CHUNKS_CANDIDATES = [
    DATA_PATH / "chroma_export.json",
    DATA_PATH / "chunks_data.json",
    BASE_PATH / "chroma_export.json",
    BASE_PATH / "chunks_data.json",
]
CHUNKS_PATH = None
for p in CHUNKS_CANDIDATES:
    if p.exists():
        CHUNKS_PATH = p
        break

# ── ChromaDB 폴더 탐색 ───────────────────────────────────────────
CHROMA_CANDIDATES = [
    "chroma_seol_qwen3",
    "chroma_team02_chunks_qwen3_emb",
    "chroma_team01_chunks_qwen3_emb",
]
CHROMA_PATH = None
for name in CHROMA_CANDIDATES:
    p = BASE_PATH / name
    if p.exists():
        CHROMA_PATH = str(p)
        break

# ── eval 파일 ────────────────────────────────────────────────────
EVAL_PATH = DATA_PATH / "pm_data.xlsx"

# ── 경로 검증 ────────────────────────────────────────────────────
print("📂 경로 탐색 결과")
print(f"  BASE_PATH  : {BASE_PATH}")
print(f"  DATA_PATH  : {DATA_PATH}")
print(f"  RESULT_DIR : {RESULT_DIR}")

if CHUNKS_PATH:
    print(f"  ✅ CHUNKS   : {CHUNKS_PATH.name}")
else:
    print(f"  ❌ CHUNKS   : 파일 없음 — 후보: {[str(p) for p in CHUNKS_CANDIDATES]}")

if CHROMA_PATH:
    print(f"  ✅ CHROMA   : {Path(CHROMA_PATH).name}")
else:
    print(f"  ❌ CHROMA   : 폴더 없음 — 후보: {CHROMA_CANDIDATES}")

if EVAL_PATH.exists():
    print(f"  ✅ EVAL     : {EVAL_PATH.name}")
else:
    print(f"  ❌ EVAL     : {EVAL_PATH} 없음")

def nfc(text: str) -> str:
    return unicodedata.normalize("NFC", str(text)) if text else ""

print("\n✅ Cell 02 완료 — 임포트 및 경로 설정 완료")

/Users/who/Desktop/code_it/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📂 경로 탐색 결과
  BASE_PATH  : /Users/who/Desktop/code_it/project01/final_files
  DATA_PATH  : /Users/who/Desktop/code_it/project01/final_files/data
  RESULT_DIR : /Users/who/Desktop/code_it/project01/final_files/results
  ✅ CHUNKS   : chroma_export.json
  ✅ CHROMA   : chroma_seol_qwen3
  ✅ EVAL     : pm_data.xlsx

✅ Cell 02 완료 — 임포트 및 경로 설정 완료


## cell 03

In [3]:
# Cell 03 — 파라미터 설정

# ── 실험 식별 ─────────────────────────────────────────────────────
NOTEBOOK_VERSION = "com_03"
PROMPT_VERSION   = "v7e"

# ── EXAONE 모델명 자동 감지 ──────────────────────────────────────
_EXAONE_CANDIDATES = [
    "exaone3.5:7.8b",
    "exaone3.5:latest",
    "hf.co/lmstudio-community/EXAONE-3.5-7.8B-Instruct-GGUF:latest",
]

def _find_exaone_model() -> str:
    try:
        from openai import OpenAI as _OAI
        _c = _OAI(base_url="http://localhost:11434/v1", api_key="ollama")
        installed = [m.id for m in _c.models.list().data]
        for m in installed:
            if "exaone" in m.lower():
                return m
        print("⚠️ EXAONE 모델을 찾지 못했습니다.")
        print(f"  설치된 모델: {installed[:10]}")
        return _EXAONE_CANDIDATES[0]
    except Exception as e:
        print(f"⚠️ Ollama 모델 목록 조회 실패: {e}")
        return _EXAONE_CANDIDATES[0]

LLM_MODEL = _find_exaone_model()

def slug_model_name(model_name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9가-힣]+", "_", str(model_name)).strip("_").lower()

EXPERIMENT_TAG = f"{slug_model_name(LLM_MODEL)}_{PROMPT_VERSION}"

# ── 기타 모델 설정 ────────────────────────────────────────────────
RAGAS_LLM_MODEL = "phi4-mini"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_MODEL  = "BAAI/bge-reranker-base"

# ── ChromaDB 컬렉션명 ─────────────────────────────────────────────
COLLECTION_NAME = "rfp_seol_chunks_qwen3"   # ← 고정

# ── 디바이스 자동 감지 ───────────────────────────────────────────
try:
    import torch
    if torch.backends.mps.is_available():
        DEVICE = "mps"
    elif torch.cuda.is_available():
        DEVICE = "cuda"
    else:
        DEVICE = "cpu"
except Exception:
    DEVICE = "cpu"

# ── Generation 파라미터 ──────────────────────────────────────────
TEMPERATURE        = 0.0
MAX_TOKENS         = 1000
MAX_CONTEXT_TOKENS = 4500

# ── Retrieval 파라미터 ───────────────────────────────────────────
RETRIEVAL_EVAL_K  = 20
ANSWER_TOP_K      = 10
DENSE_CANDIDATES  = 60
BM25_CANDIDATES   = 60
RRF_CANDIDATES    = 40

USE_RERANKER              = False
RERANKER_FALLBACK_TO_RRF  = True
RERANKER_BATCH_SIZE       = 4
RERANKER_MAX_LENGTH       = 512

# ── 검색 보정 파라미터 ───────────────────────────────────────────
NUMERIC_CHUNK_BOOST     = 0.12
AGENCY_CHUNK_BOOST      = 0.05
EVIDENCE_CHUNK_BOOST    = 0.20
BUSINESS_OVERLAP_BOOST  = 0.10
CORE_AMOUNT_BOOST       = 0.60
WRONG_BUSINESS_PENALTY  = 0.15
SOURCE_AUTOFILL_TOP_N   = 3

# ── Deterministic answer ─────────────────────────────────────────
USE_DETERMINISTIC_BUDGET_ANSWER     = True
USE_DETERMINISTIC_MULTI_AMOUNT_CALC = True
USE_DETERMINISTIC_DURATION_ANSWER   = True
DETERMINISTIC_BUDGET_MIN_SCORE      = 0.10

# ── DB 제어 ──────────────────────────────────────────────────────
FORCE_REBUILD = False
FORCE_PARSE   = False

# ── 임베딩 파라미터 ──────────────────────────────────────────────
EMBED_BATCH_SIZE      = 8
EMBED_TEXT_MAX_CHARS  = 3000
EMBED_MAX_SEQ_LENGTH  = 1024
CHROMA_ADD_BATCH_SIZE = 64

# ── 평가 설정 ────────────────────────────────────────────────────
USE_QUICK_EVAL = False        # False = 전체 500건 평가
EVAL_SAMPLE_N  = 500

# pm_data.xlsx 시트명 자동 감지
try:
    _xl          = pd.ExcelFile(EVAL_PATH)
    EVAL_SHEET_NAME = _xl.sheet_names[0]
    print(f"  pm_data.xlsx 시트명: {_xl.sheet_names}")
except Exception:
    EVAL_SHEET_NAME = None

# ── 결과 저장 경로 ───────────────────────────────────────────────
RESULT_PATH = RESULT_DIR / f"eval_results_{NOTEBOOK_VERSION}_{EXPERIMENT_TAG}.json"

print("✅ Cell 03 완료 — 파라미터 설정 완료")
print(f"  NOTEBOOK_VERSION : {NOTEBOOK_VERSION}")
print(f"  PROMPT_VERSION   : {PROMPT_VERSION}")
print(f"  EXPERIMENT_TAG   : {EXPERIMENT_TAG}")
print(f"  LLM_MODEL        : {LLM_MODEL}")
print(f"  COLLECTION_NAME  : {COLLECTION_NAME}")
print(f"  DEVICE           : {DEVICE}")
print(f"  TEMPERATURE      : {TEMPERATURE}")
print(f"  MAX_TOKENS       : {MAX_TOKENS}")
print(f"  ANSWER_TOP_K     : {ANSWER_TOP_K}")
print(f"  USE_RERANKER     : {USE_RERANKER}")
print(f"  EVAL_SHEET_NAME  : {EVAL_SHEET_NAME}")
print(f"  RESULT_PATH      : {RESULT_PATH}")

# ── 실행 전 필수 조건 확인 ──────────────────────────────────────
print("\n🔎 실행 전 필수 조건 확인")
ok = True
if not CHUNKS_PATH or not CHUNKS_PATH.exists():
    print("  ❌ chunks 파일 없음"); ok = False
else:
    print(f"  ✅ chunks 파일  : {CHUNKS_PATH.name}")
if not CHROMA_PATH or not Path(CHROMA_PATH).exists():
    print("  ❌ ChromaDB 없음"); ok = False
else:
    print(f"  ✅ ChromaDB     : {Path(CHROMA_PATH).name}")
if not EVAL_PATH.exists():
    print(f"  ❌ eval 파일 없음: {EVAL_PATH.name}"); ok = False
else:
    print(f"  ✅ eval 파일    : {EVAL_PATH.name}")
print("\n  ✅ 모든 조건 충족" if ok else "\n  ⚠️ 위 항목 해결 후 진행하세요")

  pm_data.xlsx 시트명: ['전체_500', 'TYPE_A_150', 'TYPE_B_200', 'TYPE_C_50', 'TYPE_D_50', 'TYPE_E_50', '요약']
✅ Cell 03 완료 — 파라미터 설정 완료
  NOTEBOOK_VERSION : com_03
  PROMPT_VERSION   : v7e
  EXPERIMENT_TAG   : hf_co_lmstudio_community_exaone_3_5_7_8b_instruct_gguf_latest_v7e
  LLM_MODEL        : hf.co/lmstudio-community/EXAONE-3.5-7.8B-Instruct-GGUF:latest
  COLLECTION_NAME  : rfp_seol_chunks_qwen3
  DEVICE           : mps
  TEMPERATURE      : 0.0
  MAX_TOKENS       : 1000
  ANSWER_TOP_K     : 10
  USE_RERANKER     : False
  EVAL_SHEET_NAME  : 전체_500
  RESULT_PATH      : /Users/who/Desktop/code_it/project01/final_files/results/eval_results_com_03_hf_co_lmstudio_community_exaone_3_5_7_8b_instruct_gguf_latest_v7e.json

🔎 실행 전 필수 조건 확인
  ✅ chunks 파일  : chroma_export.json
  ✅ ChromaDB     : chroma_seol_qwen3
  ✅ eval 파일    : pm_data.xlsx

  ✅ 모든 조건 충족


In [4]:
# Cell 04 — chunks 로드 (chroma_export.json 기반)

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

# chroma_export.json 구조 (ids/documents/metadatas)와
# chunks_data.json 구조 (chunk_id/text/metadata) 모두 대응
if "ids" in raw:
    # chroma_export.json 형식
    all_chunks = [
        {
            "chunk_id"  : cid,
            "text"      : doc,
            "metadata"  : meta,
            "chunk_type": meta.get("chunk_type", "normal")
        }
        for cid, doc, meta in zip(raw["ids"], raw["documents"], raw["metadatas"])
    ]
else:
    # chunks_data.json 형식
    all_chunks = raw

chunk_map = {c["chunk_id"]: c for c in all_chunks}

agency_list = sorted({
    nfc(c["metadata"].get("발주기관", ""))
    for c in all_chunks
    if c["metadata"].get("발주기관")
})

# 파일형식 / 청크 유형 집계
fmt_dist  = {}
type_dist = {}
for c in all_chunks:
    fmt  = c["metadata"].get("파일형식", "unknown")
    ct   = c.get("chunk_type", "unknown")
    fmt_dist[fmt]  = fmt_dist.get(fmt, 0) + 1
    type_dist[ct]  = type_dist.get(ct, 0) + 1

print("✅ Cell 04 완료 — chunks 로드 완료")
print(f"  파일 소스       : {CHUNKS_PATH.name}")
print(f"  총 청크 수      : {len(all_chunks):,}개")
print(f"  발주기관 수     : {len(agency_list)}개")
print(f"  파일형식 분포   : {fmt_dist}")
print(f"  chunk_type      : {type_dist}")

✅ Cell 04 완료 — chunks 로드 완료
  파일 소스       : chroma_export.json
  총 청크 수      : 51,366개
  발주기관 수     : 406개
  파일형식 분포   : {'hwp': 44598, 'pdf': 6768}
  chunk_type      : {'summary': 12439, 'normal': 35429, 'table': 3498}


In [5]:
# Cell 05 — Ollama 클라이언트 초기화

openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

def chat_ollama(
    messages: list[dict],
    model: str,
    temperature: float = 0.1,
    max_tokens: int = 800
) -> str:
    try:
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_completion_tokens=max_tokens,
        )
    except TypeError:
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
    return response.choices[0].message.content

# 연결 및 속도 테스트
try:
    t0 = time.time()
    test_resp = chat_ollama(
        messages=[{"role": "user", "content": "안녕하세요. 짧게 테스트라고만 답하세요."}],
        model=LLM_MODEL,
        temperature=0,
        max_tokens=20,
    )
    elapsed = round(time.time() - t0, 2)
    print(f"✅ Cell 05 완료 — Ollama 연결 완료")
    print(f"  모델        : {LLM_MODEL}")
    print(f"  응답 시간   : {elapsed}s")
    print(f"  응답 내용   : {test_resp[:50]}")
except Exception as e:
    print(f"⚠️ Ollama 연결 실패: {e}")
    print("  ollama serve 실행 여부 및 모델 설치 확인 필요")

✅ Cell 05 완료 — Ollama 연결 완료
  모델        : hf.co/lmstudio-community/EXAONE-3.5-7.8B-Instruct-GGUF:latest
  응답 시간   : 0.72s
  응답 내용   : 테스트 완료


In [6]:
# Cell 06 — Qwen3-Embedding 로드 + ChromaDB 로드

import torch

print("🔄 Qwen3-Embedding-0.6B 로드 중...")
emb_model = SentenceTransformer(
    EMBEDDING_MODEL,
    trust_remote_code=True,
    device=DEVICE
)

try:
    old_len = emb_model.max_seq_length
    emb_model.max_seq_length = min(old_len, EMBED_MAX_SEQ_LENGTH)
    print(f"  max_seq_length: {old_len} → {emb_model.max_seq_length}")
except Exception as e:
    print(f"  max_seq_length 설정 생략: {e}")

def prepare_embedding_text(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:EMBED_TEXT_MAX_CHARS]

def get_embeddings(texts: list[str], batch_size: int = EMBED_BATCH_SIZE) -> list[list[float]]:
    prepared = [prepare_embedding_text(t) for t in texts]
    return emb_model.encode(
        prepared,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()

print(f"✅ 임베딩 모델 로드 완료: device={DEVICE}")

# ─── ChromaDB 초기화 ────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
existing_collections = [c.name for c in chroma_client.list_collections()]

print(f"\n📂 ChromaDB 경로: {CHROMA_PATH}")

if not existing_collections:
    print("❌ ChromaDB에 컬렉션이 없습니다.")
    raise RuntimeError("ChromaDB 컬렉션 없음")

# ─── 청크 수 기준으로 가장 큰 컬렉션 선택 ──────────────────────
print(f"\n  컬렉션별 청크 수 확인 중...")
collection_counts = {}
for name in existing_collections:
    try:
        tmp   = chroma_client.get_collection(name)
        count = tmp.count()
        collection_counts[name] = count
        print(f"  - {name}: {count:,}개")
    except Exception:
        collection_counts[name] = 0
        print(f"  - {name}: 확인 실패")

# COLLECTION_NAME에 데이터가 있으면 그대로 사용
# 없으면 가장 큰 컬렉션으로 자동 전환
if collection_counts.get(COLLECTION_NAME, 0) > 0:
    print(f"\n✅ 지정 컬렉션 사용: '{COLLECTION_NAME}'")
else:
    best_name  = max(collection_counts, key=collection_counts.get)
    best_count = collection_counts[best_name]

    if best_count == 0:
        print("\n❌ 데이터가 있는 컬렉션이 없습니다.")
        print("   team_01 Cell 07을 먼저 실행하여 ChromaDB를 구축하세요.")
        raise RuntimeError("모든 컬렉션이 비어 있음")

    print(f"\n⚠️  '{COLLECTION_NAME}' 이 비어 있어 자동 전환합니다.")
    print(f"   '{COLLECTION_NAME}' (0개) → '{best_name}' ({best_count:,}개)")
    COLLECTION_NAME = best_name

# ─── 컬렉션 로드 ───────────────────────────────────────────────
collection = chroma_client.get_collection(name=COLLECTION_NAME)
count      = collection.count()

print(f"\n✅ Cell 06 완료 — ChromaDB 로드 완료")
print(f"  컬렉션명   : {COLLECTION_NAME}")
print(f"  청크 수    : {count:,}개")

if "sample" in COLLECTION_NAME:
    print(f"\n  ⚠️  현재 샘플 컬렉션({count:,}개)을 사용 중입니다.")
    print(f"  전체 구축 후에는 team_01 Cell 07 실행 → FORCE_REBUILD=False로 변경하세요.")

🔄 Qwen3-Embedding-0.6B 로드 중...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 21269.29it/s]


  max_seq_length: 32768 → 1024
✅ 임베딩 모델 로드 완료: device=mps

📂 ChromaDB 경로: /Users/who/Desktop/code_it/project01/final_files/chroma_seol_qwen3

  컬렉션별 청크 수 확인 중...
  - rfp_seol_chunks_qwen3: 51,366개

✅ 지정 컬렉션 사용: 'rfp_seol_chunks_qwen3'

✅ Cell 06 완료 — ChromaDB 로드 완료
  컬렉션명   : rfp_seol_chunks_qwen3
  청크 수    : 51,366개


In [7]:
# Cell 07 — BM25 인덱스 구축

def tokenize(text: str) -> list[str]:
    text = nfc(text).lower()
    return re.findall(r"[가-힣A-Za-z0-9]+", text)

print("⏳ BM25 인덱스 구축 중...")
bm25_corpus = [tokenize(c["text"]) for c in all_chunks]
bm25_index  = BM25Okapi(bm25_corpus)

test_q   = "나노종합기술원 용역 예산"
scores   = bm25_index.get_scores(tokenize(test_q))
top_idx  = int(np.argmax(scores))
top_name = all_chunks[top_idx]["metadata"].get("사업명", "")[:30]
print(f"  BM25 테스트: '{test_q}' → {top_name}")

print(f"✅ Cell 07 완료 — BM25 인덱스 구축 완료 ({len(bm25_corpus):,}개)")

⏳ BM25 인덱스 구축 중...
  BM25 테스트: '나노종합기술원 용역 예산' → 스마트 팹 서비스 활용체계 구축관련 설비온라인 시스템 
✅ Cell 07 완료 — BM25 인덱스 구축 완료 (51,366개)


## cell 08

In [8]:
# Cell 08 — retrieve() 정의
# prom09: prom06_fix1 기반 안정 retrieval + 약한 business 보정 + core amount 보존

reranker = None
reranker_model = None

if USE_RERANKER:
    try:
        from FlagEmbedding import FlagReranker
        reranker = FlagReranker(RERANKER_MODEL, use_fp16=False)
        print(f"✅ BGE-Reranker 로드 완료: {RERANKER_MODEL}")
    except Exception as e:
        print(f"⚠️ Reranker 로드 실패: {e}")
        print("  → RRF fallback 사용")
        reranker = None
        reranker_model = None
else:
    print("ℹ️ USE_RERANKER=False — RRF 기반 검색만 사용합니다.")

# ── Alias / Signal 설정 ──────────────────────────────────────────
AGENCY_ALIASES = {
    "가스공사": "한국가스공사",
    "한국 가스공사": "한국가스공사",
    "수자원공사": "한국수자원공사",
    "k-water": "한국수자원공사",
    "kwater": "한국수자원공사",
    "수은": "한국수출입은행",
    "수출입은행": "한국수출입은행",
    "수출입 은행": "한국수출입은행",
    "gkl": "그랜드코리아레저(주)",
    "그랜드코리아레저": "그랜드코리아레저(주)",
    "그랜드코리아레져": "그랜드코리아레저(주)",
    "그렌드코리아레져": "그랜드코리아레저(주)",
    "경희대학교산학협력단": "경희대학교산학협력단",
    "경희대학교 산학협력단": "경희대학교산학협력단",
    "굥희머학교": "경희대학교산학협력단",
    "경희대 산학협력단": "경희대학교산학협력단",
    "경희대학교": "경희대학교",
    "경희대": "경희대학교",
    "인천공항운영서비스": "인천공항운영서비스(주)",
    "인천공항운영서비스㈜": "인천공항운영서비스(주)",
    "인천공항운영서비스 주": "인천공항운영서비스(주)",
    "서울시립대": "서울시립대학교",
    "서울시륍대": "서울시립대학교",
    "남서울대": "남서울대학교",
    "남서율대햑교": "남서울대학교",
    "gist": "광주과학기술원",
    "지스트": "광주과학기술원",
}

NUMERIC_QUERY_WORDS = [
    "예산", "금액", "사업비", "소요예산", "총사업비", "용역금액",
    "용역예산", "발주금액", "추정가격", "기초금액", "배정액",
    "원", "만원", "억 원", "억원", "억", "%", "비율", "차액", "합계", "총액", "얼마"
]

CORE_AMOUNT_WORDS = [
    "용역금액", "용역예산", "사업예산", "사업금액", "사업비",
    "총사업비", "소요예산", "기초금액", "추정가격", "배정예산",
    "배정액", "계약금액"
]

LOW_PRIORITY_AMOUNT_WORDS = [
    "입찰가격", "평가", "평점", "배점", "실적", "유사사업",
    "수행실적", "평가기준", "가격평가", "정량평가", "정성평가",
    "참여율", "지분율", "하도급", "보증금율"
]

BUDGET_TEXT_WORDS = CORE_AMOUNT_WORDS + LOW_PRIORITY_AMOUNT_WORDS

REGION_QUERY_WORDS = ["과업 대상 지역", "대상 지역", "대상지역", "지자체", "어느 지역", "지역"]
REGION_TEXT_WORDS = [
    "과업 대상 지역", "대상지역", "대상 지역", "과업대상", "과업 대상",
    "대상 지", "위치", "용인시", "성남시", "광주시", "하남시", "지자체",
    "공급 지역", "급수구역", "계획 공급 지역"
]

DURATION_QUERY_WORDS = ["기간", "개월", "일정", "안정화", "추진 일정", "전체 몇 개월"]
DURATION_TEXT_WORDS = [
    "사업기간", "용역기간", "계약체결일", "착수일", "개월",
    "안정화", "추진일정", "정식 오픈", "구축 및 개발", "통합테스트"
]

EQUIPMENT_QUERY_WORDS = ["서버", "장비", "인프라", "스토리지", "SAN", "DB", "AP"]
EQUIPMENT_TEXT_WORDS = [
    "ECR", "시스템 장비", "인프라", "서버", "스토리지", "SAN", "스위치",
    "S/4 HANA", "S/4HANA", "DB 서버", "AP 서버", "백업", "LDM", "외부연계",
    "운영 DB", "품질 DB", "BW DB", "웹서버", "WAS"
]

MONEY_REGEX = re.compile(
    r"\d[\d,]*(?:\.\d+)?\s*(?:원|천원|만\s*원|만원|백만\s*원|백만원|억\s*원|억원|억|%|퍼센트)"
)

STOP_TERMS = {
    "한국", "사업", "용역", "구축", "시스템", "프로젝트", "예산", "금액", "얼마",
    "전체", "관련", "이번", "연도", "추진", "발주", "타당성", "조사", "기본계획",
    "수립", "정보", "운영", "기능", "개선", "고도화", "기간", "과업", "대상",
    "지역", "포함", "정확히", "대략", "무엇", "어떤", "주요", "신규"
}

def normalize_for_agency_match(text: str) -> str:
    text = nfc(str(text)).lower()
    text = re.sub(r"[\s_()\[\]{}·\-.,/「」『』'\"“”‘’㈜]", "", text)
    return text

def normalize_for_keyword(text: str) -> str:
    text = nfc(str(text)).lower()
    text = re.sub(r"[\s_()\[\]{}·\-.,/「」『』'\"“”‘’㈜]", "", text)
    return text

def resolve_agency_name(name: str) -> str | None:
    if not name:
        return None

    target_norm = normalize_for_agency_match(name)

    for agency in agency_list:
        if normalize_for_agency_match(agency) == target_norm:
            return agency

    for agency in agency_list:
        agency_norm = normalize_for_agency_match(agency)
        if target_norm and (target_norm in agency_norm or agency_norm in target_norm):
            return agency

    return None

def detect_agencies_from_query(query: str) -> list[str]:
    q_norm = normalize_for_agency_match(query)
    found = []

    for alias, canonical in AGENCY_ALIASES.items():
        alias_norm = normalize_for_agency_match(alias)
        if alias_norm and alias_norm in q_norm:
            resolved = resolve_agency_name(canonical)
            if resolved:
                found.append(resolved)

    for agency in agency_list:
        agency_norm = normalize_for_agency_match(agency)
        if agency_norm and agency_norm in q_norm:
            found.append(agency)

    # 더 구체적인 기관명 우선: 산학협력단 vs 대학교 본교 혼동 방지
    unique = []
    seen = set()

    found = sorted(found, key=lambda x: len(normalize_for_agency_match(x)), reverse=True)

    for agency in found:
        key = normalize_for_agency_match(agency)

        # 이미 더 긴 기관명이 포함되어 있으면 짧은 기관명은 제외
        if any(key in normalize_for_agency_match(u) and key != normalize_for_agency_match(u) for u in unique):
            continue

        if key not in seen:
            unique.append(agency)
            seen.add(key)

    return unique

def detect_agency_from_query(query: str) -> str | None:
    agencies = detect_agencies_from_query(query)
    return agencies[0] if agencies else None

def has_any_word(text: str, words: list[str]) -> bool:
    t = str(text).lower()
    return any(w.lower() in t for w in words)

def is_numeric_query(query: str) -> bool:
    q = nfc(str(query))
    return any(w in q for w in NUMERIC_QUERY_WORDS) or bool(MONEY_REGEX.search(q))

def chunk_text_pool(chunk: dict) -> str:
    meta = chunk.get("metadata", {}) or {}
    return "\n".join([
        str(chunk.get("text", "")),
        str(chunk.get("numeric_summary", "")),
        str(meta.get("numeric_summary", "")),
        str(meta.get("section", "")),
        str(meta.get("사업명", "")),
        str(meta.get("발주기관", "")),
        str(meta.get("파일명", "")),
    ])

def is_core_amount_line(text: str) -> bool:
    return bool(MONEY_REGEX.search(str(text))) and has_any_word(text, CORE_AMOUNT_WORDS)

def is_low_priority_amount_line(text: str) -> bool:
    return bool(MONEY_REGEX.search(str(text))) and has_any_word(text, LOW_PRIORITY_AMOUNT_WORDS)

def chunk_has_core_amount_signal(chunk: dict) -> bool:
    return is_core_amount_line(chunk_text_pool(chunk))

def chunk_has_money_signal(chunk: dict) -> bool:
    text = chunk_text_pool(chunk)
    return bool(MONEY_REGEX.search(text)) and has_any_word(text, BUDGET_TEXT_WORDS)

def extract_business_terms(query: str) -> list[str]:
    q = nfc(str(query))
    quoted = re.findall(r"[\"'‘’“”](.*?)[\"'‘’“”]", q)
    terms = []

    for item in quoted:
        for t in re.findall(r"[가-힣A-Za-z0-9+]+", item):
            if len(t) >= 2 and t not in STOP_TERMS:
                terms.append(t)

    for t in re.findall(r"[가-힣A-Za-z0-9+]+", q):
        if len(t) >= 2 and t not in STOP_TERMS:
            terms.append(t)

    agency_norms = [normalize_for_keyword(a) for a in detect_agencies_from_query(q)]

    cleaned = []
    seen = set()

    for t in terms:
        nt = normalize_for_keyword(t)
        if not nt:
            continue

        if any(nt in a or a in nt for a in agency_norms):
            continue

        if nt not in seen:
            cleaned.append(t)
            seen.add(nt)

    return cleaned[:20]

def business_overlap_score(query: str, chunk: dict) -> float:
    terms = extract_business_terms(query)

    if not terms:
        return 0.0

    meta = chunk.get("metadata", {}) or {}
    target_text = " ".join([
        str(meta.get("사업명", "")),
        str(meta.get("파일명", "")),
        str(meta.get("section", "")),
        str(chunk.get("text", ""))[:500],
    ])

    target_norm = normalize_for_keyword(target_text)

    hit = 0
    for t in terms:
        nt = normalize_for_keyword(t)
        if nt and nt in target_norm:
            hit += 1

    return hit / max(len(terms), 1)

def is_probably_wrong_business(query: str, chunk: dict) -> bool:
    terms = extract_business_terms(query)

    if len(terms) < 3:
        return False

    agencies = detect_agencies_from_query(query)
    if not agencies:
        return False

    meta = chunk.get("metadata", {}) or {}
    chunk_agency = normalize_for_agency_match(meta.get("발주기관", ""))
    agency_hit = any(chunk_agency == normalize_for_agency_match(a) for a in agencies)

    if not agency_hit:
        return False

    # core amount는 Q021/Q022 때문에 살림
    if chunk_has_core_amount_signal(chunk):
        return False

    return business_overlap_score(query, chunk) < 0.08

def get_query_signal_types(query: str) -> set[str]:
    q = nfc(str(query))
    signals = set()

    if is_numeric_query(q):
        signals.add("numeric")
    if has_any_word(q, REGION_QUERY_WORDS):
        signals.add("region")
    if has_any_word(q, DURATION_QUERY_WORDS):
        signals.add("duration")
    if has_any_word(q, EQUIPMENT_QUERY_WORDS):
        signals.add("equipment")

    return signals

def chunk_matches_query_signal(query: str, chunk: dict) -> bool:
    signals = get_query_signal_types(query)
    text = chunk_text_pool(chunk)

    if "numeric" in signals and chunk_has_money_signal(chunk):
        return True
    if "region" in signals and has_any_word(text, REGION_TEXT_WORDS):
        return True
    if "duration" in signals and has_any_word(text, DURATION_TEXT_WORDS):
        return True
    if "equipment" in signals and has_any_word(text, EQUIPMENT_TEXT_WORDS):
        return True

    return False

def make_source(meta: dict) -> str:
    발주기관 = meta.get("발주기관", "")
    사업명   = meta.get("사업명", "")
    파일명   = meta.get("파일명", "")
    section  = meta.get("section", "")
    page     = meta.get("page", "")

    loc = f"§{section}" if section else (f"p.{page}" if page else "")
    loc_str = f" | {loc}" if loc else ""

    if 발주기관 or 사업명:
        return f"[{발주기관} — {사업명}{loc_str}]"
    return f"[{파일명}{loc_str}]"

def apply_contextual_boost(query: str, chunk: dict, score: float, agency_filter: str | None = None) -> float:
    boosted = float(score)

    overlap = business_overlap_score(query, chunk)
    if overlap > 0:
        boosted += BUSINESS_OVERLAP_BOOST * overlap

    if chunk_matches_query_signal(query, chunk):
        boosted += EVIDENCE_CHUNK_BOOST

    if is_numeric_query(query) and chunk_has_core_amount_signal(chunk):
        boosted += CORE_AMOUNT_BOOST

    elif is_numeric_query(query) and chunk_has_money_signal(chunk):
        boosted += NUMERIC_CHUNK_BOOST

    if agency_filter:
        meta = chunk.get("metadata", {}) or {}
        if normalize_for_agency_match(meta.get("발주기관", "")) == normalize_for_agency_match(agency_filter):
            boosted += AGENCY_CHUNK_BOOST

    if is_probably_wrong_business(query, chunk):
        boosted -= WRONG_BUSINESS_PENALTY

    return boosted

def reciprocal_rank_fusion(rankings: list[list[str]], k: int = 60) -> dict[str, float]:
    scores = {}

    for ranking in rankings:
        for rank, chunk_id in enumerate(ranking, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)

    return scores

def safe_rerank(query: str, candidates: list[dict]) -> list[tuple[dict, float]] | None:
    return None

def _retrieve_single(
    query: str,
    top_k: int = RETRIEVAL_EVAL_K,
    agency_filter: str | None = None,
) -> list[dict]:
    query = nfc(str(query))

    where_filter = {"발주기관": agency_filter} if agency_filter else None
    q_emb = get_embeddings([query])[0]
    dense_n = min(DENSE_CANDIDATES, collection.count())

    dense_kwargs = {
        "query_embeddings": [q_emb],
        "n_results": dense_n,
        "include": ["documents", "metadatas", "distances"],
    }

    if where_filter:
        available = sum(
            1 for c in all_chunks
            if normalize_for_agency_match(c.get("metadata", {}).get("발주기관", "")) == normalize_for_agency_match(agency_filter)
        )

        if available <= 0:
            return []

        dense_kwargs["where"] = where_filter
        dense_kwargs["n_results"] = max(1, min(dense_n, available))

    dense_res = collection.query(**dense_kwargs)
    dense_ids = dense_res.get("ids", [[]])[0]

    bm25_scores = bm25_index.get_scores(tokenize(query))

    if agency_filter:
        a_norm = normalize_for_agency_match(agency_filter)
        filtered_idx = [
            i for i, c in enumerate(all_chunks)
            if normalize_for_agency_match(c.get("metadata", {}).get("발주기관", "")) == a_norm
        ]
    else:
        filtered_idx = list(range(len(all_chunks)))

    bm25_ranked = sorted(
        [(i, bm25_scores[i]) for i in filtered_idx],
        key=lambda x: x[1],
        reverse=True,
    )[:BM25_CANDIDATES]

    bm25_ids = [all_chunks[i]["chunk_id"] for i, _ in bm25_ranked]

    rrf_scores = reciprocal_rank_fusion([dense_ids, bm25_ids])
    candidate_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:RRF_CANDIDATES]

    candidates = [chunk_map[cid] for cid in candidate_ids if cid in chunk_map]

    ranked = sorted(
        [(c, rrf_scores.get(c["chunk_id"], 0.0)) for c in candidates],
        key=lambda x: x[1],
        reverse=True,
    )

    adjusted = []
    for chunk, score in ranked:
        new_score = apply_contextual_boost(query, chunk, score, agency_filter)
        adjusted.append((chunk, new_score))

    adjusted = sorted(adjusted, key=lambda x: x[1], reverse=True)

    results = []
    for chunk, score in adjusted[:top_k]:
        item = dict(chunk)
        item["metadata"] = dict(chunk.get("metadata", {}) or {})
        item["score"] = round(float(score), 6)
        item["source"] = make_source(item["metadata"])
        item["business_overlap"] = round(business_overlap_score(query, item), 4)
        item["is_core_amount"] = bool(chunk_has_core_amount_signal(item))
        results.append(item)

    return results

def find_evidence_chunks_from_same_docs(
    query: str,
    retrieved_chunks: list[dict],
    max_add: int = 8,
) -> list[dict]:
    """
    prom06_fix1 성향 유지:
    검색된 관련 문서 내부에서 signal chunk만 보강.
    단, prom08의 core amount 보존은 유지.
    """
    if not retrieved_chunks:
        return []

    target_files = []
    for c in retrieved_chunks[:12]:
        meta = c.get("metadata", {}) or {}
        fname = meta.get("파일명", "")
        if fname and fname not in target_files:
            target_files.append(fname)

    evidence = []

    for fname in target_files:
        same_doc_chunks = [
            c for c in all_chunks
            if c.get("metadata", {}).get("파일명", "") == fname
        ]

        scored = []

        for c in same_doc_chunks:
            text = chunk_text_pool(c)
            cc = dict(c)
            cc["metadata"] = dict(c.get("metadata", {}) or {})
            cc["source"] = make_source(cc["metadata"])

            score = 0.0

            if is_numeric_query(query) and chunk_has_core_amount_signal(c):
                score += 6.0

            elif is_numeric_query(query) and chunk_has_money_signal(c):
                score += 2.0

            if has_any_word(query, REGION_QUERY_WORDS) and has_any_word(text, REGION_TEXT_WORDS):
                score += 4.0

            if has_any_word(query, DURATION_QUERY_WORDS) and has_any_word(text, DURATION_TEXT_WORDS):
                score += 4.0

            if has_any_word(query, EQUIPMENT_QUERY_WORDS) and has_any_word(text, EQUIPMENT_TEXT_WORDS):
                score += 4.0

            score += business_overlap_score(query, cc) * 1.5

            if is_probably_wrong_business(query, cc):
                score -= 0.3

            if score > 0:
                cc["score"] = round(score, 6)
                cc["business_overlap"] = round(business_overlap_score(query, cc), 4)
                cc["is_core_amount"] = bool(chunk_has_core_amount_signal(c))
                scored.append(cc)

        scored = sorted(scored, key=lambda x: x.get("score", 0), reverse=True)
        evidence.extend(scored[:3])

    seen = set()
    unique = []

    for c in evidence:
        cid = c.get("chunk_id")
        if cid and cid not in seen:
            unique.append(c)
            seen.add(cid)

    return unique[:max_add]

def merge_retrieved_lists(lists: list[list[dict]], top_k: int) -> list[dict]:
    merged = []
    seen = set()

    max_len = max((len(lst) for lst in lists), default=0)

    for i in range(max_len):
        for lst in lists:
            if i >= len(lst):
                continue

            item = lst[i]
            cid = item.get("chunk_id")

            if cid and cid not in seen:
                merged.append(item)
                seen.add(cid)

            if len(merged) >= top_k:
                return merged

    return merged[:top_k]

def retrieve(query: str, top_k: int = RETRIEVAL_EVAL_K, verbose: bool = False) -> list[dict]:
    query = nfc(str(query))
    agencies = detect_agencies_from_query(query)

    if verbose:
        print(f"  감지된 발주기관: {agencies if agencies else '없음'}")
        print(f"  signal types: {get_query_signal_types(query)}")
        print(f"  business terms: {extract_business_terms(query)}")

    result_sets = []

    if len(agencies) >= 2:
        for agency in agencies:
            result_sets.append(
                _retrieve_single(query, top_k=max(top_k, ANSWER_TOP_K), agency_filter=agency)
            )

        result_sets.append(_retrieve_single(query, top_k=top_k, agency_filter=None))
        base = merge_retrieved_lists(result_sets, top_k=top_k)

    elif len(agencies) == 1:
        base = _retrieve_single(query, top_k=top_k, agency_filter=agencies[0])

        if not base:
            base = _retrieve_single(query, top_k=top_k, agency_filter=None)

    else:
        base = _retrieve_single(query, top_k=top_k, agency_filter=None)

    evidence = find_evidence_chunks_from_same_docs(query, base, max_add=8)
    final = merge_retrieved_lists([evidence, base], top_k=top_k)

    return final

print("\n  retrieve() 테스트 중...")
_test = retrieve(
    "한국수자원공사의 용인 첨단 시스템반도체 국가산단 용수공급사업과 한국수출입은행의 모잠비크 마푸토 ITS 구축사업의 예산 차액은 얼마입니까?",
    verbose=True
)
print(f"  반환 청크 수 : {len(_test)}개")
if _test:
    print(f"  Top-1 기관   : {_test[0]['metadata'].get('발주기관','')}")
    print(f"  Top-1 사업명 : {_test[0]['metadata'].get('사업명','')[:60]}")
    print(f"  Top-1 score  : {_test[0].get('score')}")
    print(f"  Top-1 core   : {_test[0].get('is_core_amount')}")
    print(f"  Top-1 overlap: {_test[0].get('business_overlap')}")

print("\n✅ Cell 08 완료 — retrieve() 정의 완료")
print(f"  USE_RERANKER          : {USE_RERANKER}")
print(f"  RETRIEVAL_EVAL_K      : {RETRIEVAL_EVAL_K}")
print(f"  ANSWER_TOP_K          : {ANSWER_TOP_K}")
print(f"  BUSINESS_OVERLAP_BOOST: {BUSINESS_OVERLAP_BOOST}")
print(f"  CORE_AMOUNT_BOOST     : {CORE_AMOUNT_BOOST}")
print(f"  WRONG_BUSINESS_PENALTY: {WRONG_BUSINESS_PENALTY}")

ℹ️ USE_RERANKER=False — RRF 기반 검색만 사용합니다.

  retrieve() 테스트 중...
  감지된 발주기관: ['한국수자원공사', '한국수출입은행']
  signal types: {'numeric'}
  business terms: ['용인', '첨단', '시스템반도체', '국가산단', '용수공급사업과', '모잠비크', '마푸토', 'ITS', '구축사업의', '차액은', '얼마입니까']
  반환 청크 수 : 20개
  Top-1 기관   : 한국수출입은행
  Top-1 사업명 : (긴급) 국가결산체계 개편에 따른 국가회계시스템 고도화
  Top-1 score  : 6.0
  Top-1 core   : True
  Top-1 overlap: 0.0

✅ Cell 08 완료 — retrieve() 정의 완료
  USE_RERANKER          : False
  RETRIEVAL_EVAL_K      : 20
  ANSWER_TOP_K          : 10
  BUSINESS_OVERLAP_BOOST: 0.1
  CORE_AMOUNT_BOOST     : 0.6
  WRONG_BUSINESS_PENALTY: 0.15


## cell 09

In [9]:
# Cell 09 — SYSTEM_PROMPT 정의
# prom07 / v7e: v7d + 금액 라벨 우선 + 기간 추정 금지 + 미기재 금액 생성 금지

SYSTEM_PROMPT_V7E = """당신은 RFP(입찰 공고) 문서 전문 분석가입니다.
반드시 아래 규칙을 따른다.

[최우선 원칙]
1. 답변은 반드시 제공된 컨텍스트와 질문에 직접 제시된 수치만 사용한다.
2. 컨텍스트에 없는 문서 사실은 추측하지 않는다.
3. 질문 자체에 금액, 비율, 수량이 명시되어 있고 단순 산술 계산만 요구하는 경우에는 그 수치를 사용하여 계산할 수 있다.
4. 단, 문서에 존재해야만 알 수 있는 사실은 반드시 컨텍스트에 근거가 있어야 한다.
5. 출력 형식은 항상 **답변**, **근거**, **출처** 세 구역을 유지한다.

[거부 규칙]
컨텍스트에 질문이 요구하는 문서 기반 정보가 없으면:
- **답변** 첫 줄을 반드시 "제공된 문서에서 해당 정보를 찾을 수 없습니다."로 시작한다.
- 그래도 **근거**, **출처** 구역은 생략하지 않는다.
- **근거**에는 어떤 문서는 검색되었고, 어떤 핵심 정보가 없었는지 간단히 적는다.
- **출처**에는 확인한 문서의 발주기관, 사업명, 파일명을 적는다.
- 일반 지식, 상식, 추정, 가능성 표현으로 보완하지 않는다.

[추정 금지]
다음 표현은 사용하지 않는다:
- "가능성이 높습니다"
- "추정됩니다"
- "일반적으로"
- "대부분의 경우"
- "추가 확인이 필요합니다" 단독 표현
- "공식 문서를 참조해야 합니다" 단독 표현

문서에 있으면 문서 내용을 답하고, 없으면 없다고 답한다.

[예산/금액/수치 인식 규칙]
다음 표현은 예산 또는 금액의 핵심 후보로 본다:
- 용역금액
- 용역예산
- 사업예산
- 사업금액
- 사업비
- 총사업비
- 소요예산
- 기초금액
- 추정가격
- 배정예산
- 배정액
- 계약금액

금액 질문에서는 위 표현이 포함된 줄을 가장 우선한다.
입찰가격 평가식, 실적 기준 금액, 배점 기준 금액, 유사사업 수행실적 금액은 해당 사업의 예산 답변 후보보다 후순위로 본다.
특히 "100분의 80", "100분의 70", "5억원 이상 실적", "1억원 이상 ~ 2억원 미만" 같은 표현은 평가 기준일 수 있으므로, 용역금액/사업예산으로 직접 답하지 않는다.

[금액 미기재/비공개 규칙]
문서에 예산이 미기재, 비공개, 확인 불가라고 되어 있으면 금액을 생성하지 않는다.
다른 유사 사업의 금액, 입찰 기준 금액, 실적 기준 금액, 평가 기준 금액을 해당 사업 예산으로 대체하지 않는다.
질문 대상 사업의 예산이 명시되지 않았으면 "제공된 문서에서 해당 사업의 예산 금액을 찾을 수 없습니다."라고 답한다.

[질문 제시 수치 계산 규칙]
질문 자체에 구체적인 금액, 비율, 수량이 명시된 경우:
1. 해당 수치는 컨텍스트 확인 없이 산술 계산에 사용할 수 있다.
2. 이 경우 계산 결과 옆에 "(질문 제시 수치 기반)"이라고 표기한다.
3. 다만 질문 속 수치가 어떤 문서의 공식 예산인지 검증해야 하는 질문이면 컨텍스트 근거가 필요하다.
4. 질문이 "A는 1억, B는 1.5억일 때 합계는?"처럼 수치를 직접 제공하면 합산할 수 있다.

[수치 계산 규칙]
1. 계산 전 원문 수치 또는 질문 제시 수치를 먼저 명시한다.
2. 계산식을 간단히 제시한다.
3. 분수는 반드시 정확히 변환한다.
   - 2분의 1 = 0.5 = 50%
   - 3분의 1 = 약 0.333 = 33.3%
   - 3분의 2 = 약 0.667 = 66.7%
4. 단위는 가능한 한 원(₩) 기준으로 통일한다.
5. 억원, 만원 표기는 괄호로 원화 병기한다.
6. 계산 결과는 반올림 여부를 명확히 한다.
7. 계산식은 실제 연산과 일치해야 한다. 예: 3분의 1은 ×(1/3), 3분의 2는 ×(2/3)이다.

[기간 질문 규칙]
기간 질문에서는 문서에 명시된 개월 수, 시작일, 종료일만 사용한다.
"상반기", "오픈 이후", "안정화 활동", "일반적으로" 같은 표현을 임의로 개월 수로 환산하지 않는다.
명시 기간이 없으면 계산하지 않는다.
사업기간, 용역기간, 구축기간, 안정화 기간이 서로 다르면 항목별로 구분한다.

[다중 기관/다중 사업 비교 규칙]
두 개 이상의 기관 또는 사업을 비교, 합산, 차액 계산하는 질문에서는:
1. 각 기관/사업별로 컨텍스트에 정보가 있는지 먼저 분리한다.
2. 다음 형식을 사용한다.
   - [기관/사업 A]&#58; 확인된 정보 또는 "해당 정보 없음"
   - [기관/사업 B]&#58; 확인된 정보 또는 "해당 정보 없음"
3. 모든 기관/사업의 필요한 정보가 컨텍스트에 있거나 질문에 직접 수치가 제시된 경우에만 계산한다.
4. 일부 기관/사업의 정보가 없으면 차액/합산 계산을 수행하지 않는다.
5. 일부 정보만 있으면 어떤 정보가 있고 어떤 정보가 없는지 명확히 말한다.

[문서 검토 규칙]
1. 컨텍스트의 [정답 후보 요약]을 먼저 확인한다.
2. 그 다음 [문서 1]부터 [문서 N]까지 검토한다.
3. 하나의 문서만 보고 성급히 답하지 않는다.
4. 핵심 용어, 기술명, 기관명, 사업명, 항목명은 원문 표현을 우선 사용한다.
5. 문서 제목만 맞고 본문에 답이 없으면 답이 있다고 보지 않는다.
6. 질문의 사업명과 다른 사업 문서의 금액을 답으로 사용하지 않는다.

[출력 형식]
반드시 아래 형식만 사용한다. 제목에 콜론을 붙이지 않는다.

**답변**
질문에 직접 답한다. 계산이 있으면 원문 수치, 계산식, 결과를 포함한다.

**근거**
[문서 N] 형식으로 근거를 적는다. 거부 시에도 확인한 문서와 누락된 정보를 적는다.

**출처**
발주기관 — 사업명 | 파일명
출처를 특정할 수 없으면 "해당 없음"이라고 적는다.
"""

SYSTEM_PROMPT_MAP = {
    "v7e": SYSTEM_PROMPT_V7E,
}

SYSTEM_PROMPT = SYSTEM_PROMPT_MAP.get(PROMPT_VERSION, SYSTEM_PROMPT_V7E)

print("✅ Cell 09 완료 — SYSTEM_PROMPT 정의 완료")
print(f"  사용 버전  : {PROMPT_VERSION}")
print(f"  프롬프트   : {len(SYSTEM_PROMPT)}자")

✅ Cell 09 완료 — SYSTEM_PROMPT 정의 완료
  사용 버전  : v7e
  프롬프트   : 2732자


## cell 10

In [10]:
# Cell 10 — ask() + context 구성 + deterministic answer
# prom09: 단순 예산 + 다중 금액 차액 + 기간 deterministic answer

def convert_cheonwon(text: str) -> str:
    def replacer(m):
        raw = m.group(1).replace(",", "")
        won = int(raw) * 1000
        orig = m.group(1)
        return f"{orig}천원 ({won:,}원)"
    return re.sub(r"([\d,]+)\s*천원", replacer, str(text))

def normalize_answer_format(answer: str) -> str:
    text = str(answer).strip()

    text = re.sub(r"\*\*\s*답변\s*:?\s*\*\*", "**답변**", text)
    text = re.sub(r"\*\*\s*근거\s*:?\s*\*\*", "**근거**", text)
    text = re.sub(r"\*\*\s*출처\s*:?\s*\*\*", "**출처**", text)

    if "**답변**" not in text:
        text = f"**답변**\n{text}"

    if "**근거**" not in text:
        text += "\n\n**근거**\n해당 없음"

    if "**출처**" not in text:
        text += "\n\n**출처**\n해당 없음"

    return text.strip()

def extract_section_local(text: str, name: str) -> str:
    pattern = rf"\*\*\s*{re.escape(name)}\s*:?\s*\*\*\s*\n?(.*?)(?=\n\s*\*\*\s*(?:답변|근거|출처)\s*:?\s*\*\*|\Z)"
    m = re.search(pattern, str(text), re.DOTALL)
    return m.group(1).strip() if m else ""

def replace_section_local(text: str, name: str, new_content: str) -> str:
    ans = extract_section_local(text, "답변")
    gnd = extract_section_local(text, "근거")
    src = extract_section_local(text, "출처")

    if name == "답변":
        ans = new_content
    elif name == "근거":
        gnd = new_content
    elif name == "출처":
        src = new_content

    return f"**답변**\n{ans}\n\n**근거**\n{gnd}\n\n**출처**\n{src}".strip()

def is_missing_section_content(content: str) -> bool:
    c = str(content).strip()
    return c in ("", "해당 없음", "없음", "—", "-")

def parse_money_to_won_local(text: str) -> list[int]:
    text = str(text)
    values = []

    patterns = [
        (r"(\d[\d,]*(?:\.\d+)?)\s*억\s*원", 100_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*억원", 100_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*억", 100_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*백만\s*원", 1_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*백만원", 1_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*만\s*원", 10_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*만원", 10_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*천\s*원", 1_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*천원", 1_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*원", 1),
    ]

    for pat, unit in patterns:
        for m in re.finditer(pat, text):
            raw = m.group(1).replace(",", "")
            try:
                values.append(int(round(float(raw) * unit)))
            except Exception:
                pass

    return sorted(set(values))

def format_won_amount(won: int) -> str:
    won = int(won)

    if won >= 100_000_000:
        eok = won / 100_000_000
        return f"{won:,}원 (약 {eok:,.2f}억 원)"

    if won >= 10_000:
        man = won / 10_000
        return f"{won:,}원 (약 {man:,.0f}만 원)"

    return f"{won:,}원"

def extract_question_numbers(query: str) -> list[str]:
    patterns = [
        r"\d[\d,]*(?:\.\d+)?\s*억\s*원?",
        r"\d[\d,]*(?:\.\d+)?\s*억",
        r"\d[\d,]*(?:\.\d+)?\s*만\s*원",
        r"\d[\d,]*(?:\.\d+)?\s*만원",
        r"\d[\d,]*(?:\.\d+)?\s*천\s*원",
        r"\d[\d,]*(?:\.\d+)?\s*천원",
        r"\d[\d,]*(?:\.\d+)?\s*원",
        r"\d[\d,]*(?:\.\d+)?\s*%",
        r"\d+\s*분의\s*\d+",
    ]

    found = []
    for p in patterns:
        found.extend(re.findall(p, str(query)))

    return list(dict.fromkeys([x.strip() for x in found if x.strip()]))

def line_priority(query: str, line: str) -> int:
    line = str(line)

    if bool(MONEY_REGEX.search(line)) and has_any_word(line, ["용역금액", "용역예산"]):
        return 0

    if bool(MONEY_REGEX.search(line)) and has_any_word(line, ["사업예산", "사업금액", "사업비", "소요예산"]):
        return 1

    if bool(MONEY_REGEX.search(line)) and has_any_word(line, ["총사업비", "기초금액", "추정가격", "배정예산", "배정액", "계약금액"]):
        return 2

    if has_any_word(line, REGION_TEXT_WORDS):
        return 3

    if has_any_word(line, DURATION_TEXT_WORDS):
        return 4

    if has_any_word(line, EQUIPMENT_TEXT_WORDS):
        return 5

    if bool(MONEY_REGEX.search(line)) and has_any_word(line, LOW_PRIORITY_AMOUNT_WORDS):
        return 9

    if bool(MONEY_REGEX.search(line)):
        return 6

    return 8

def extract_candidate_lines(query: str, text: str, max_lines: int = 10) -> list[str]:
    text = str(text)
    raw_lines = re.split(r"[\n\r]+", text)
    candidates = []

    for line in raw_lines:
        line = re.sub(r"\s+", " ", line).strip()
        if not line:
            continue

        ok = False

        if is_numeric_query(query) and (MONEY_REGEX.search(line) or has_any_word(line, BUDGET_TEXT_WORDS)):
            ok = True

        if has_any_word(query, REGION_QUERY_WORDS) and has_any_word(line, REGION_TEXT_WORDS):
            ok = True

        if has_any_word(query, DURATION_QUERY_WORDS) and has_any_word(line, DURATION_TEXT_WORDS):
            ok = True

        if has_any_word(query, EQUIPMENT_QUERY_WORDS) and has_any_word(line, EQUIPMENT_TEXT_WORDS):
            ok = True

        if ok:
            candidates.append(line)

    if not candidates:
        compact = re.sub(r"\s+", " ", text).strip()

        if is_numeric_query(query) and (MONEY_REGEX.search(compact) or has_any_word(compact, BUDGET_TEXT_WORDS)):
            candidates.append(compact[:600])
        elif has_any_word(query, REGION_QUERY_WORDS) and has_any_word(compact, REGION_TEXT_WORDS):
            candidates.append(compact[:600])
        elif has_any_word(query, DURATION_QUERY_WORDS) and has_any_word(compact, DURATION_TEXT_WORDS):
            candidates.append(compact[:600])
        elif has_any_word(query, EQUIPMENT_QUERY_WORDS) and has_any_word(compact, EQUIPMENT_TEXT_WORDS):
            candidates.append(compact[:600])

    candidates = list(dict.fromkeys([c for c in candidates if c]))

    candidates = sorted(
        candidates,
        key=lambda x: (line_priority(query, x), -len(x))
    )

    return candidates[:max_lines]

def build_evidence_card(query: str, context_docs: list[dict]) -> str:
    cards = []
    qnums = extract_question_numbers(query)

    if qnums:
        cards.append(
            "[질문에 직접 제시된 수치]\n"
            + "\n".join(f"- {n}" for n in qnums)
        )

    for i, doc in enumerate(context_docs, start=1):
        meta = doc.get("metadata", {}) or {}

        text = "\n".join([
            str(doc.get("numeric_summary", "")),
            str(meta.get("numeric_summary", "")),
            str(doc.get("text", "")),
        ])

        lines = extract_candidate_lines(query, text, max_lines=10)

        if lines:
            cards.append(
                f"[문서 {i} 핵심 후보]\n"
                f"발주기관: {meta.get('발주기관', '')}\n"
                f"사업명: {meta.get('사업명', '')}\n"
                f"파일명: {meta.get('파일명', '')}\n"
                + "\n".join(f"- {line}" for line in lines[:10])
            )

    if not cards:
        return ""

    return "[정답 후보 요약]\n" + "\n\n".join(cards)

def is_simple_budget_query(query: str) -> bool:
    q = nfc(str(query))

    budget_words = ["예산", "금액", "사업비", "소요 예산", "소요예산", "용역금액", "얼마"]
    block_words = [
        "차액", "합계", "더한", "합산", "총 자금", "상회", "비율", "%",
        "3분의", "2분의", "1/3", "2/3", "기간", "개월", "지역", "지자체",
        "서버", "장비", "인프라", "둘 중", "비교", "명시된 사업"
    ]

    return any(w in q for w in budget_words) and not any(w in q for w in block_words)

def is_multi_amount_calc_query(query: str) -> bool:
    q = nfc(str(query))
    calc_words = ["차액", "합계", "합산", "더한", "총액", "총 자금"]
    return is_numeric_query(q) and any(w in q for w in calc_words)

def is_duration_query(query: str) -> bool:
    return has_any_word(query, DURATION_QUERY_WORDS)

def get_target_agency_norms(query: str) -> set[str]:
    return {normalize_for_agency_match(a) for a in detect_agencies_from_query(query)}

def extract_core_amount_candidates(query: str, answer_chunks: list[dict]) -> list[dict]:
    candidates = []
    target_agencies = get_target_agency_norms(query)

    for doc_idx, doc in enumerate(answer_chunks, start=1):
        meta = doc.get("metadata", {}) or {}
        agency_norm = normalize_for_agency_match(meta.get("발주기관", ""))

        if target_agencies and agency_norm not in target_agencies:
            continue

        if is_probably_wrong_business(query, doc):
            continue

        text = "\n".join([
            str(doc.get("numeric_summary", "")),
            str(meta.get("numeric_summary", "")),
            str(doc.get("text", "")),
        ])

        for line in re.split(r"[\n\r]+", text):
            line = re.sub(r"\s+", " ", line).strip()
            if not line:
                continue

            if not is_core_amount_line(line):
                continue

            if is_low_priority_amount_line(line):
                continue

            won_values = parse_money_to_won_local(line)
            won_values = [v for v in won_values if v >= 10_000]

            if not won_values:
                continue

            overlap = business_overlap_score(query, doc)
            priority = line_priority(query, line)

            score = 0.0
            score += max(0, 10 - priority)
            score += overlap * 5
            score += float(doc.get("score", 0)) * 0.1

            candidates.append({
                "doc_idx": doc_idx,
                "line": line,
                "won": max(won_values),
                "score": score,
                "source": make_source(meta),
                "agency": meta.get("발주기관", ""),
                "business": meta.get("사업명", ""),
                "filename": meta.get("파일명", ""),
                "chunk_id": doc.get("chunk_id", ""),
                "business_overlap": overlap,
            })

    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)

    unique = []
    seen = set()

    for c in candidates:
        key = (c["agency"], c["business"], c["won"])
        if key not in seen:
            unique.append(c)
            seen.add(key)

    return unique

def try_build_deterministic_budget_answer(query: str, answer_chunks: list[dict]) -> str | None:
    if not USE_DETERMINISTIC_BUDGET_ANSWER:
        return None

    if not is_simple_budget_query(query):
        return None

    candidates = extract_core_amount_candidates(query, answer_chunks)

    if not candidates:
        return None

    best = candidates[0]

    if best["score"] < 1.0:
        return None

    amount_text = format_won_amount(best["won"])

    return (
        f"**답변**\n"
        f"{best['business']}의 예산은 **{amount_text}**입니다.\n\n"
        f"**근거**\n"
        f"[문서 {best['doc_idx']}] {best['line']}\n\n"
        f"**출처**\n"
        f"{best['agency']} — {best['business']} | {best['filename']}"
    )

def try_build_deterministic_multi_amount_answer(query: str, answer_chunks: list[dict]) -> str | None:
    if not USE_DETERMINISTIC_MULTI_AMOUNT_CALC:
        return None

    if not is_multi_amount_calc_query(query):
        return None

    candidates = extract_core_amount_candidates(query, answer_chunks)

    if len(candidates) < 2:
        return None

    # 기관별 대표 금액 1개
    agency_best = {}
    for c in candidates:
        key = c["agency"] or c["business"]
        if key not in agency_best or c["score"] > agency_best[key]["score"]:
            agency_best[key] = c

    selected = list(agency_best.values())

    if len(selected) < 2:
        return None

    selected = sorted(selected, key=lambda x: x["score"], reverse=True)[:2]

    a, b = selected[0], selected[1]

    if "차액" in query:
        diff = abs(a["won"] - b["won"])
        answer_line = (
            f"{a['business']}의 금액은 {format_won_amount(a['won'])}, "
            f"{b['business']}의 금액은 {format_won_amount(b['won'])}입니다. "
            f"두 금액의 차액은 **{format_won_amount(diff)}**입니다."
        )
        calc_line = f"{a['won']:,}원 - {b['won']:,}원 = {diff:,}원"
    else:
        total = a["won"] + b["won"]
        answer_line = (
            f"{a['business']}의 금액은 {format_won_amount(a['won'])}, "
            f"{b['business']}의 금액은 {format_won_amount(b['won'])}입니다. "
            f"두 금액의 합계는 **{format_won_amount(total)}**입니다."
        )
        calc_line = f"{a['won']:,}원 + {b['won']:,}원 = {total:,}원"

    return (
        f"**답변**\n"
        f"{answer_line}\n\n"
        f"**근거**\n"
        f"[문서 {a['doc_idx']}] {a['line']}\n"
        f"[문서 {b['doc_idx']}] {b['line']}\n"
        f"계산식: {calc_line}\n\n"
        f"**출처**\n"
        f"{a['agency']} — {a['business']} | {a['filename']}\n"
        f"{b['agency']} — {b['business']} | {b['filename']}"
    )

def try_build_deterministic_duration_answer(query: str, answer_chunks: list[dict]) -> str | None:
    if not USE_DETERMINISTIC_DURATION_ANSWER:
        return None

    if not is_duration_query(query):
        return None

    duration_patterns = [
        r"(?:사업기간|용역기간|구축기간|수행기간)\s*[:：]?\s*([0-9]+)\s*개월",
        r"총\s*([0-9]+)\s*개월",
        r"([0-9]+)\s*개월",
    ]

    candidates = []

    for doc_idx, doc in enumerate(answer_chunks, start=1):
        meta = doc.get("metadata", {}) or {}
        text = str(doc.get("text", ""))

        if is_probably_wrong_business(query, doc):
            continue

        lines = re.split(r"[\n\r]+", text)

        for line in lines:
            clean = re.sub(r"\s+", " ", line).strip()
            if not clean:
                continue

            if not has_any_word(clean, DURATION_TEXT_WORDS):
                continue

            for pat in duration_patterns:
                m = re.search(pat, clean)
                if m:
                    months = int(m.group(1))
                    candidates.append({
                        "months": months,
                        "line": clean,
                        "doc_idx": doc_idx,
                        "agency": meta.get("발주기관", ""),
                        "business": meta.get("사업명", ""),
                        "filename": meta.get("파일명", ""),
                        "score": business_overlap_score(query, doc) + float(doc.get("score", 0)) * 0.01,
                    })

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)
    best = candidates[0]

    return (
        f"**답변**\n"
        f"{best['business']}의 명시된 기간은 **{best['months']}개월**입니다.\n\n"
        f"**근거**\n"
        f"[문서 {best['doc_idx']}] {best['line']}\n\n"
        f"**출처**\n"
        f"{best['agency']} — {best['business']} | {best['filename']}"
    )

def filter_chunks_by_target_agency(query: str, chunks: list[dict]) -> list[dict]:
    agencies = detect_agencies_from_query(query)

    if not agencies:
        return chunks

    target_norms = {normalize_for_agency_match(a) for a in agencies}

    kept = []
    removed = []

    for c in chunks:
        meta = c.get("metadata", {}) or {}
        agency_norm = normalize_for_agency_match(meta.get("발주기관", ""))

        if agency_norm in target_norms:
            kept.append(c)
        else:
            removed.append(c)

    if len(kept) >= 3:
        return kept + removed

    return chunks

def select_answer_chunks(query: str, retrieved_chunks: list[dict], top_k: int = ANSWER_TOP_K) -> list[dict]:
    if not retrieved_chunks:
        return []

    chunks = filter_chunks_by_target_agency(query, retrieved_chunks)

    priority = []
    seen = set()

    def add_chunk(c):
        cid = c.get("chunk_id")
        if cid and cid not in seen:
            priority.append(c)
            seen.add(cid)

    agencies = detect_agencies_from_query(query)

    # 다중 기관 질문: 기관별 후보를 먼저 1개씩 확보
    if len(agencies) >= 2:
        for agency in agencies:
            a_norm = normalize_for_agency_match(agency)
            agency_chunks = [
                c for c in chunks
                if normalize_for_agency_match(c.get("metadata", {}).get("발주기관", "")) == a_norm
            ]
            agency_chunks = sorted(
                agency_chunks,
                key=lambda c: (
                    not chunk_has_core_amount_signal(c),
                    -business_overlap_score(query, c),
                    -float(c.get("score", 0))
                )
            )

            if agency_chunks:
                add_chunk(agency_chunks[0])

    # numeric 질문: core amount 보존
    if is_numeric_query(query):
        core_chunks = [c for c in chunks if chunk_has_core_amount_signal(c)]
        core_chunks = sorted(
            core_chunks,
            key=lambda c: (-business_overlap_score(query, c), is_probably_wrong_business(query, c), -float(c.get("score", 0)))
        )
        for c in core_chunks[:5]:
            add_chunk(c)

    # signal chunks
    signal_chunks = [c for c in chunks if chunk_matches_query_signal(query, c)]
    signal_chunks = sorted(
        signal_chunks,
        key=lambda c: (
            -business_overlap_score(query, c),
            is_probably_wrong_business(query, c),
            -float(c.get("score", 0))
        )
    )
    for c in signal_chunks[:5]:
        add_chunk(c)

    # 기존 순서 유지
    for c in chunks:
        add_chunk(c)
        if len(priority) >= top_k:
            break

    return priority[:top_k]

def build_context_text(
    context_docs: list[dict],
    query: str = "",
    max_chars: int = MAX_CONTEXT_TOKENS * 3,
) -> tuple[str, str]:
    parts = []
    total = 0

    evidence_card = build_evidence_card(query, context_docs)
    if evidence_card:
        parts.append(evidence_card)

    for i, doc in enumerate(context_docs, start=1):
        meta = doc.get("metadata", {}) or {}
        numeric = doc.get("numeric_summary") or meta.get("numeric_summary", "")

        header = (
            f"[문서 {i}]\n"
            f"출처: {doc.get('source', make_source(meta))}\n"
            f"발주기관: {meta.get('발주기관', '')}\n"
            f"사업명: {meta.get('사업명', '')}\n"
            f"파일명: {meta.get('파일명', '')}\n"
        )

        if numeric:
            header += f"수치요약: {numeric}\n"

        block = header + "본문:\n" + str(doc.get("text", ""))

        if total + len(block) > max_chars:
            remain = max_chars - total
            if remain > 300:
                parts.append(block[:remain])
            break

        parts.append(block)
        total += len(block)

    return "\n\n".join(parts), evidence_card

def make_source_block(answer_chunks: list[dict], top_n: int = SOURCE_AUTOFILL_TOP_N) -> str:
    rows = []
    seen = set()

    for c in answer_chunks[:top_n]:
        meta = c.get("metadata", {}) or {}

        agency = meta.get("발주기관", "")
        biz = meta.get("사업명", "")
        fname = meta.get("파일명", "")

        row = f"{agency} — {biz} | {fname}".strip()

        if row and row not in seen:
            rows.append(row)
            seen.add(row)

    return "\n".join(rows) if rows else "해당 없음"

def make_ground_block_from_evidence(evidence_card: str, max_lines: int = 5) -> str:
    if not evidence_card:
        return "해당 없음"

    lines = []
    for line in evidence_card.splitlines():
        line = line.strip()
        if line.startswith("- "):
            lines.append(line)

    if not lines:
        return "해당 없음"

    return "\n".join(lines[:max_lines])

def autofill_ground_and_source(answer: str, answer_chunks: list[dict], evidence_card: str) -> str:
    text = normalize_answer_format(answer)

    ground = extract_section_local(text, "근거")
    if is_missing_section_content(ground):
        text = replace_section_local(text, "근거", make_ground_block_from_evidence(evidence_card))

    source = extract_section_local(text, "출처")
    if is_missing_section_content(source):
        text = replace_section_local(text, "출처", make_source_block(answer_chunks))

    return text

def ask(query: str, history: list[dict] | None = None) -> dict:
    if history is None:
        history = []

    retrieved_chunks = retrieve(query, top_k=RETRIEVAL_EVAL_K)
    answer_chunks = select_answer_chunks(query, retrieved_chunks, top_k=ANSWER_TOP_K)

    context_text, evidence_card = build_context_text(answer_chunks, query=query)

    deterministic_answer = None
    deterministic_type = None

    # 다중 금액 계산이 단순 예산보다 먼저 와야 함
    deterministic_answer = try_build_deterministic_multi_amount_answer(query, answer_chunks)
    if deterministic_answer is not None:
        deterministic_type = "multi_amount"

    if deterministic_answer is None:
        deterministic_answer = try_build_deterministic_budget_answer(query, answer_chunks)
        if deterministic_answer is not None:
            deterministic_type = "budget"

    if deterministic_answer is None:
        deterministic_answer = try_build_deterministic_duration_answer(query, answer_chunks)
        if deterministic_answer is not None:
            deterministic_type = "duration"

    if deterministic_answer is not None:
        answer = deterministic_answer
        used_deterministic = True
    else:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        messages += history
        messages.append({
            "role": "user",
            "content": f"[참고 문서]\n{context_text}\n\n[질문]\n{query}",
        })

        answer = chat_ollama(
            messages=messages,
            model=LLM_MODEL,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        used_deterministic = False

    answer = convert_cheonwon(answer)
    answer = normalize_answer_format(answer)
    answer = autofill_ground_and_source(answer, answer_chunks, evidence_card)

    new_history = history + [
        {"role": "user", "content": query},
        {"role": "assistant", "content": answer},
    ]

    if len(new_history) > 6:
        new_history = new_history[-6:]

    return {
        "query": query,
        "answer": answer,
        "contexts": [d.get("text", "") for d in answer_chunks],
        "chunks": retrieved_chunks,
        "answer_chunks": answer_chunks,
        "evidence_card": evidence_card,
        "used_deterministic": used_deterministic,
        "deterministic_type": deterministic_type,
        "history": new_history,
    }

print("✅ Cell 10 완료 — ask() 정의 완료")
print(f"  retrieve 반환: {RETRIEVAL_EVAL_K}개")
print(f"  LLM context  : {ANSWER_TOP_K}개")
print(f"  deterministic budget answer: {USE_DETERMINISTIC_BUDGET_ANSWER}")
print(f"  deterministic multi amount : {USE_DETERMINISTIC_MULTI_AMOUNT_CALC}")
print(f"  deterministic duration     : {USE_DETERMINISTIC_DURATION_ANSWER}")

✅ Cell 10 완료 — ask() 정의 완료
  retrieve 반환: 20개
  LLM context  : 10개
  deterministic budget answer: True
  deterministic multi amount : True
  deterministic duration     : True


## cell 11

In [11]:
# Cell 11 — ConversationManager / V2
# prom07: rule-based rewrite 유지 + 오타 보정 보강

AMBIGUOUS_PATTERNS = [
    "그 ", "그럼", "그중", "그것", "해당", "그 부분",
    "거기서", "위의", "앞서", "방금", "거기", "그쪽",
    "그 연장선상", "말씀하신", "연장선상"
]

TYPO_HINTS = [
    "굥희", "머학교", "샨합", "혁렵", "증뵤", "시쓰탬",
    "샤업", "에싼", "그렌드", "레져", "쥬", "츄진",
    "구룹", "씨스탬", "구쭉", "입너", "서울시륍",
    "죙단", "하눈", "남서율", "대햑교", "즁보", "운용"
]

TYPO_REPLACEMENTS = {
    "굥희머학교": "경희대학교",
    "굥희": "경희",
    "머학교": "대학교",
    "샨합혁렵댠": "산학협력단",
    "샨합": "산학",
    "혁렵": "협력",
    "댠": "단",
    "증뵤시쓰탬": "정보시스템",
    "증뵤": "정보",
    "시쓰탬": "시스템",
    "씨스탬": "시스템",
    "샤업": "사업",
    "에싼": "예산",
    "에산": "예산",
    "얼마져": "얼마입니까",
    "잇쟌아효": "",
    "잇쟌아": "",
    "그거": "",
    "그렌드코리아레져": "그랜드코리아레저",
    "그렌드": "그랜드",
    "레져": "레저",
    "쥬": "주",
    "츄진": "추진",
    "구룹웨에": "그룹웨어",
    "구룹웨어": "그룹웨어",
    "구쭉": "구축",
    "입너가": "입니까",
    "입너": "입니",
    "서울시륍대": "서울시립대학교",
    "죙단분석": "종단분석",
    "하눈": "하는",
    "남서율대햑교": "남서울대학교",
    "대햑교": "대학교",
    "즁보": "정보",
    "운용": "운영",
}

REWRITE_SYSTEM_PROMPT = """
당신은 RFP 검색 쿼리 최적화 전문가입니다.

역할:
1. 모호한 지시어가 포함된 질문은 대화 히스토리를 참고하여 독립 질문으로 재작성한다.
2. 오타, 비문, 발음식 표기가 포함된 질문은 정확한 한국어로 교정한다.
3. 발주기관명, 사업명, 시스템명은 검색에 유리하도록 공식 명칭에 가깝게 복원한다.
4. 질문에 금액, 예산, 기간, 수량, 비율 등이 있으면 그 조건을 유지한다.
5. 재작성된 질문 한 문장만 출력한다.

출력 규칙:
- 반드시 질문 한 문장만 출력한다.
- 설명, 답변, 근거, 출처를 쓰지 않는다.
"""

def rule_based_query_cleanup(query: str) -> str:
    text = nfc(str(query))

    for wrong, right in sorted(TYPO_REPLACEMENTS.items(), key=lambda x: len(x[0]), reverse=True):
        text = text.replace(wrong, right)

    text = re.sub(r"\s+", " ", text).strip()

    # 남은 구어체 정리
    text = re.sub(r"^그\s+", "", text).strip()
    text = text.replace("있잖아요", "")
    text = text.replace("잇잖아요", "")
    text = re.sub(r"\s+", " ", text).strip()

    return text

def is_ambiguous_or_typo(query: str) -> bool:
    q = nfc(str(query))

    if any(p in q for p in AMBIGUOUS_PATTERNS):
        return True

    if any(t in q for t in TYPO_HINTS):
        return True

    if re.search(r"[ㄱ-ㅎㅏ-ㅣ]", q):
        return True

    cleaned = rule_based_query_cleanup(q)
    if cleaned != q:
        return True

    return False

def clean_rewritten_query(text: str, fallback: str) -> str:
    text = nfc(str(text)).strip()

    if not text:
        return fallback

    invalid = ["**답변**", "**근거**", "**출처**", "답변:", "근거:", "출처:"]
    if any(m in text for m in invalid):
        return fallback

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return fallback

    text = lines[0]
    text = re.sub(r"^[\"'“”‘’]+|[\"'“”‘’]+$", "", text).strip()

    if len(text) > 400:
        return fallback

    return text if text else fallback

def rewrite_query(query: str, history: list[dict]) -> str:
    corrected = rule_based_query_cleanup(query)

    # 명확한 오타 보정이 있었으면 LLM rewrite보다 rule-based 결과를 우선 사용
    if corrected != query and len(corrected) >= 8:
        return corrected

    if not is_ambiguous_or_typo(query):
        return query

    messages = [{"role": "system", "content": REWRITE_SYSTEM_PROMPT}]

    if history:
        messages.append({
            "role": "user",
            "content": "이전 대화 히스토리입니다. 후속 질문의 지시어 해소에만 참고하세요.\n"
                       + "\n".join([f"{h['role']}: {h['content']}" for h in history[-4:]])
        })

    messages.append({
        "role": "user",
        "content": (
            "다음 질문을 RFP 문서 검색에 적합한 독립 질문 한 문장으로 재작성하세요.\n"
            f"원문 질문: {query}\n"
            f"1차 보정 질문: {corrected}"
        ),
    })

    try:
        rewritten = chat_ollama(
            messages=messages,
            model=LLM_MODEL,
            temperature=0.0,
            max_tokens=120,
        )
        return clean_rewritten_query(rewritten, corrected)

    except Exception as e:
        print(f"⚠️ rewrite 실패 — rule-based 보정 질문 사용: {e}")
        return corrected

class ConversationManager:
    def __init__(self):
        self.history = []

    def ask(self, query: str) -> dict:
        result = ask(query, history=self.history)
        self.history = result["history"]
        result["original_query"] = query
        result["rewritten_query"] = query
        return result

    def reset(self):
        self.history = []

class ConversationManagerV2(ConversationManager):
    def ask(self, query: str) -> dict:
        rewritten = rewrite_query(query, self.history)
        result = ask(rewritten, history=self.history)
        result["original_query"] = query
        result["rewritten_query"] = rewritten
        self.history = result["history"]
        return result

print("✅ Cell 11 완료 — ConversationManager / V2 정의 완료")
print("  오타 질문은 rule-based rewrite 우선 적용")

✅ Cell 11 완료 — ConversationManager / V2 정의 완료
  오타 질문은 rule-based rewrite 우선 적용


In [12]:
# Cell 12 — EVAL_SET 로드 (pm_data.xlsx)
import ast

REFUSAL_KEYWORDS = [
    "찾을 수 없", "확인되지 않", "없습니다",
    "제공된 문서에서", "알 수 없",
    "명시되어 있지 않",
    "포함되어 있지 않",
    "정보가 없",
]

QUICK_EVAL_IDS = [
    "Q021", "Q022", "Q023", "Q186",
    "Q107", "Q289", "Q328",
    "Q008", "Q389",
    "Q136", "Q176",
    "Q058", "Q077",
    "Q039", "Q179",
]

def is_empty_value(v) -> bool:
    if v is None: return True
    try:
        if pd.isna(v): return True
    except Exception: pass
    return isinstance(v, str) and v.strip() == ""

def parse_json_like(value, default):
    if is_empty_value(value): return default
    if isinstance(value, (list, dict)): return value
    text = str(value).strip()
    try: return ast.literal_eval(text)
    except Exception: pass
    try: return json.loads(text)
    except Exception: pass
    return default

# ── pm_data.xlsx 로드 ────────────────────────────────────────────
df_eval = pd.read_excel(
    EVAL_PATH,
    sheet_name = EVAL_SHEET_NAME
)

# 컬럼명 정규화 (공백 제거)
df_eval.columns = [c.strip() for c in df_eval.columns]
print(f"컬럼 목록: {df_eval.columns.tolist()}")
print(f"총 {len(df_eval)}건 로드")

# ── EVAL_SET 구성 ─────────────────────────────────────────────────
EVAL_SET = []
for idx, row in df_eval.iterrows():
    q_id = str(row.get("질문ID", row.get("ID", f"Q{idx+1:03d}"))).strip()

    # 근거 문서 파싱
    raw_docs = row.get("근거 문서", row.get("근거문서", ""))
    source_docs = parse_json_like(raw_docs, [])
    if isinstance(source_docs, str) and source_docs:
        source_docs = [d.strip() for d in source_docs.split(",") if d.strip()]

    # 파일명_stem (확장자 제거)
    stems = []
    for d in source_docs:
        stem = re.sub(r"\.(hwp|pdf)$", "", str(d).strip(), flags=re.IGNORECASE)
        stems.append(stem)

    q_type = str(row.get("유형", "A")).strip().upper()
    question = str(row.get("질문", "")).strip()

    EVAL_SET.append({
        "id"          : q_id,
        "question"    : question,
        "type"        : q_type,
        "source_docs" : source_docs,
        "파일명_stems": stems,
        "gold_answer" : str(row.get("정답", row.get("gold_answer", ""))).strip(),
    })

# ── Quick eval 서브셋 ────────────────────────────────────────────
QUICK_SUBSET  = [e for e in EVAL_SET if e["id"] in QUICK_EVAL_IDS]
EVAL_SET_TO_RUN = QUICK_SUBSET if USE_QUICK_EVAL else EVAL_SET[:EVAL_SAMPLE_N]

# 유형 분포
type_dist_eval = {}
for e in EVAL_SET:
    t = e["type"]
    type_dist_eval[t] = type_dist_eval.get(t, 0) + 1

print(f"\n✅ Cell 12 완료 — EVAL_SET 로드 완료")
print(f"  전체 평가셋     : {len(EVAL_SET)}건")
print(f"  실행 대상       : {len(EVAL_SET_TO_RUN)}건 ({'Quick' if USE_QUICK_EVAL else 'Full'})")
print(f"  유형 분포       : {type_dist_eval}")
print(f"  근거문서 있음   : {sum(1 for e in EVAL_SET if e['source_docs'])}건")

컬럼 목록: ['ID', '유형', '난이도', '질문', '정답', '근거 문서', '메타필터', '대화 이력', '소스 페이지', '추론 과정', '검증 포인트']
총 500건 로드

✅ Cell 12 완료 — EVAL_SET 로드 완료
  전체 평가셋     : 500건
  실행 대상       : 500건 (Full)
  유형 분포       : {'A': 150, 'B': 200, 'C': 50, 'D': 50, 'E': 50}
  근거문서 있음   : 500건


## cell 13

In [13]:
# Cell 13 — run_eval() + 시각화 함수
# prom07: answer_chunks / evidence_card 디버깅 출력 추가

def parse_answer(answer: str) -> str:
    text = str(answer)
    m = re.search(
        r"\*\*\s*답변\s*:?\s*\*\*\s*\n?(.*?)(?=\n\s*\*\*\s*(?:근거|출처)\s*:?\s*\*\*|\Z)",
        text,
        re.DOTALL,
    )
    return m.group(1).strip() if m else text.strip()

def get_section(text: str, name: str) -> str:
    pattern = rf"\*\*\s*{re.escape(name)}\s*:?\s*\*\*\s*\n?(.*?)(?=\n\s*\*\*\s*(?:답변|근거|출처)\s*:?\s*\*\*|\Z)"
    m = re.search(pattern, str(text), re.DOTALL)
    return m.group(1).strip() if m else "—"

def summarize_retrieved_docs(chunks: list[dict]) -> list[dict]:
    docs = []

    for c in chunks:
        meta = c.get("metadata", {}) or {}
        docs.append({
            "chunk_id": c.get("chunk_id", ""),
            "공고번호": meta.get("공고번호", ""),
            "사업명": meta.get("사업명", ""),
            "발주기관": meta.get("발주기관", ""),
            "파일명": meta.get("파일명", ""),
            "파일형식": meta.get("파일형식", ""),
            "page": meta.get("page", ""),
            "section": meta.get("section", ""),
            "source": c.get("source", ""),
            "score": c.get("score", None),
            "business_overlap": c.get("business_overlap", None),
            "is_core_amount": c.get("is_core_amount", None),
        })

    return docs

def summarize_answer_chunks(chunks: list[dict]) -> list[dict]:
    return summarize_retrieved_docs(chunks)

def compact_evidence_card(evidence_card: str, max_chars: int = 600) -> str:
    text = str(evidence_card or "").strip()
    if not text:
        return "—"

    text = re.sub(r"\n{3,}", "\n\n", text)
    return text[:max_chars] + ("..." if len(text) > max_chars else "")

def display_result(idx: int, total: int, item: dict, result: dict, elapsed: float):
    qid = item["id"]
    qtype = item["type"]
    question = item["question"]
    answer = result.get("answer", "")
    retrieved = result.get("retrieved_docs", [])
    answer_docs = result.get("answer_docs", [])
    evidence_card = result.get("evidence_card", "")
    error = result.get("error")
    rewritten_query = result.get("rewritten_query", question)

    type_colors = {
        "A": "#3b82f6",
        "B": "#06b6d4",
        "C": "#8b5cf6",
        "D": "#6b7280",
        "E": "#f59e0b",
    }

    tc = type_colors.get(qtype, "#6b7280")
    ans = get_section(answer, "답변")
    gnd = get_section(answer, "근거")
    src = get_section(answer, "출처")

    parsed = parse_answer(answer)
    is_refusal = any(kw in parsed for kw in REFUSAL_KEYWORDS)
    is_refusal_expected = item.get("is_refusal_expected", False)

    refusal_badge = ""
    if is_refusal_expected:
        refusal_badge = (
            '<span style="background:#22c55e;color:white;padding:2px 6px;border-radius:4px;font-size:0.75em;margin-left:6px">✅ 거부 성공</span>'
            if is_refusal else
            '<span style="background:#ef4444;color:white;padding:2px 6px;border-radius:4px;font-size:0.75em;margin-left:6px">❌ 거부 실패</span>'
        )

    false_refusal_badge = ""
    if (not is_refusal_expected) and is_refusal:
        false_refusal_badge = '<span style="background:#f97316;color:white;padding:2px 6px;border-radius:4px;font-size:0.75em;margin-left:6px">⚠️ 과잉 거부</span>'

    one_char_warn = ""
    if ans in ("—", "") or len(ans) <= 1:
        one_char_warn = '<span style="background:#ef4444;color:white;padding:2px 6px;border-radius:4px;font-size:0.75em;margin-left:6px">⚠️ 1자 답변</span>'

    source_warn = ""
    if src in ("—", "", "해당 없음"):
        source_warn = '<span style="background:#64748b;color:white;padding:2px 6px;border-radius:4px;font-size:0.75em;margin-left:6px">출처 비어있음</span>'

    rewrite_html = ""
    if rewritten_query != question:
        rewrite_html = f'''
        <div style="margin-top:4px;font-size:0.75em;color:#7c3aed">
            🔁 Rewrite: {html_lib.escape(rewritten_query)}
        </div>
        '''

    docs_html = ""
    if retrieved:
        rows = "".join([
            f'<div style="font-size:0.8em;color:#374151;padding:1px 0">'
            f'{i+1}. {html_lib.escape(d.get("발주기관","—")[:20])} — {html_lib.escape(d.get("사업명","—")[:45])}'
            f'<span style="color:#9ca3af;margin-left:6px">score={d.get("score","")}, ov={d.get("business_overlap","")}, core={d.get("is_core_amount","")}</span></div>'
            for i, d in enumerate(retrieved[:5])
        ])
        docs_html = f'''
        <div style="margin-top:8px">
            <div style="font-size:0.75em;color:#6b7280">📂 검색 문서 상위 5개</div>
            {rows}
        </div>
        '''

    answer_docs_html = ""
    if answer_docs:
        rows = "".join([
            f'<div style="font-size:0.8em;color:#1f2937;padding:1px 0">'
            f'{i+1}. {html_lib.escape(d.get("발주기관","—")[:20])} — {html_lib.escape(d.get("사업명","—")[:45])}'
            f'<span style="color:#9ca3af;margin-left:6px">score={d.get("score","")}, ov={d.get("business_overlap","")}, core={d.get("is_core_amount","")}</span></div>'
            for i, d in enumerate(answer_docs[:7])
        ])
        answer_docs_html = f'''
        <div style="margin-top:8px;background:#f8fafc;border-radius:6px;padding:6px">
            <div style="font-size:0.75em;color:#475569">🧩 LLM 입력 answer_chunks 상위 7개</div>
            {rows}
        </div>
        '''

    evidence_html = ""
    if evidence_card:
        ev = html_lib.escape(compact_evidence_card(evidence_card))
        evidence_html = f'''
        <div style="margin-top:8px;background:#fffbeb;border:1px solid #fde68a;border-radius:6px;padding:6px">
            <div style="font-size:0.75em;color:#92400e">🧾 Evidence Card</div>
            <pre style="white-space:pre-wrap;font-size:0.74em;color:#78350f;margin:4px 0 0 0">{ev}</pre>
        </div>
        '''

    if error:
        body = f'<div style="color:#ef4444">❌ 오류: {html_lib.escape(str(error))}</div>'
    else:
        body = f'''
        <div style="margin-bottom:8px">
            <span style="font-size:0.75em;color:#6b7280">❓ 질문 </span>
            <span style="font-size:0.88em;font-weight:500">{html_lib.escape(question)}</span>
            {rewrite_html}
        </div>
        <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px">
            <div>
                <div style="font-size:0.72em;color:#6b7280">📄 답변</div>
                <div style="font-size:0.83em;line-height:1.5">{html_lib.escape(ans[:450])}{"..." if len(ans) > 450 else ""}</div>
            </div>
            <div>
                <div style="font-size:0.72em;color:#6b7280">📎 근거 / 출처</div>
                <div style="font-size:0.8em;color:#4b5563">{html_lib.escape(gnd[:220])}{"..." if len(gnd) > 220 else ""}</div>
                <div style="font-size:0.72em;color:#9ca3af;margin-top:3px">🔖 {html_lib.escape(src[:120])}</div>
            </div>
        </div>
        {docs_html}
        {answer_docs_html}
        {evidence_html}
        '''

    html_str = f'''
    <div style="border:1px solid #e5e7eb;border-radius:8px;padding:12px;margin:5px 0;background:#fff">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;
                    padding-bottom:6px;border-bottom:1px solid #f3f4f6">
            <div>
                <span style="font-weight:700">[{idx}/{total}] {html_lib.escape(qid)}</span>
                <span style="background:{tc};color:white;padding:2px 7px;border-radius:4px;font-size:0.75em;margin-left:6px">
                    Type {html_lib.escape(qtype)}</span>
                {refusal_badge}
                {false_refusal_badge}
                {one_char_warn}
                {source_warn}
                <span style="color:#9ca3af;font-size:0.72em;margin-left:6px">{html_lib.escape(item.get("difficulty",""))}</span>
            </div>
            <span style="color:#6b7280;font-size:0.8em">⏱ {elapsed}s</span>
        </div>
        {body}
    </div>
    '''

    display(HTML(html_str))

def run_eval(eval_set: list[dict], use_rewrite: bool = False) -> list[dict]:
    label = "V2_Rewrite" if use_rewrite else "V1_Baseline"
    results = []

    display(HTML(f'''
    <div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:6px;padding:10px;margin:8px 0">
        <b>▶ {label} 평가 시작</b> — {len(eval_set)}개 | LLM: {LLM_MODEL} | PROMPT: {PROMPT_VERSION}
    </div>
    '''))

    for idx, item in enumerate(eval_set, start=1):
        qtype = item["type"]
        manager = ConversationManagerV2() if use_rewrite else ConversationManager()
        manager.reset()

        if qtype == "C" and item.get("history"):
            manager.history = copy.deepcopy(item["history"])

        start = time.time()

        try:
            result = manager.ask(item["question"])
            elapsed_sec = round(time.time() - start, 2)

            entry = {
                "id": item["id"],
                "type": qtype,
                "difficulty": item.get("difficulty", ""),
                "question": item["question"],
                "answer": result.get("answer", ""),
                "parsed_answer": parse_answer(result.get("answer", "")),
                "contexts": result.get("contexts", []),
                "retrieved_docs": summarize_retrieved_docs(result.get("chunks", [])),
                "answer_docs": summarize_answer_chunks(result.get("answer_chunks", [])),
                "evidence_card": result.get("evidence_card", ""),
                "ground_truth": item.get("ground_truth", ""),
                "evidence_docs": item.get("relevant_docs", []),
                "is_refusal_expected": item.get("is_refusal_expected", False),
                "rewritten_query": result.get("rewritten_query", item["question"]),
                "elapsed_sec": elapsed_sec,
                "error": None,
            }

        except Exception as e:
            elapsed_sec = round(time.time() - start, 2)

            entry = {
                "id": item["id"],
                "type": qtype,
                "difficulty": item.get("difficulty", ""),
                "question": item["question"],
                "answer": "",
                "parsed_answer": "",
                "contexts": [],
                "retrieved_docs": [],
                "answer_docs": [],
                "evidence_card": "",
                "ground_truth": item.get("ground_truth", ""),
                "evidence_docs": item.get("relevant_docs", []),
                "is_refusal_expected": item.get("is_refusal_expected", False),
                "rewritten_query": item["question"],
                "elapsed_sec": elapsed_sec,
                "error": str(e),
            }

        results.append(entry)
        display_result(idx, len(eval_set), item, entry, elapsed_sec)

    errors = sum(1 for r in results if r["error"])
    avg_time = round(sum(r["elapsed_sec"] for r in results) / max(len(results), 1), 2)

    one_char = sum(
        1 for r in results
        if r.get("parsed_answer", "") in ("—", "") or len(r.get("parsed_answer", "")) <= 1
    )

    refusal_ok = sum(
        1 for r in results
        if r.get("is_refusal_expected") and any(kw in r.get("parsed_answer", "") for kw in REFUSAL_KEYWORDS)
    )

    refusal_total = sum(1 for r in results if r.get("is_refusal_expected"))

    false_refusal = sum(
        1 for r in results
        if (not r.get("is_refusal_expected")) and any(kw in r.get("parsed_answer", "") for kw in REFUSAL_KEYWORDS)
    )

    source_missing = sum(
        1 for r in results
        if get_section(r.get("answer", ""), "출처") in ("", "—", "해당 없음")
    )

    display(HTML(f'''
    <div style="background:#f0fdf4;border:1px solid #bbf7d0;border-radius:6px;padding:10px;margin:8px 0">
        ✅ <b>{label} 완료</b> | 평균: <b>{avg_time}s</b> |
        1자 답변: <b>{one_char}개</b> |
        거부 성공: <b>{refusal_ok}/{refusal_total}</b> |
        과잉 거부: <b>{false_refusal}개</b> |
        출처 누락: <b>{source_missing}개</b> |
        오류: <b>{errors}개</b>
    </div>
    '''))

    return results

print("✅ Cell 13 완료 — run_eval() 정의 완료")

✅ Cell 13 완료 — run_eval() 정의 완료


## cell 14

In [ ]:
# Cell 14 — 평가 실행
# ※ USE_QUICK_EVAL=True: 15개 서브셋 (~6분)
# ※ USE_QUICK_EVAL=False: 전체 50개 (~20분)

print(f"🔄 평가 시작 — {EXPERIMENT_TAG} | {len(EVAL_SET_TO_RUN)}개 질문")
print(f"  PROMPT_VERSION : {PROMPT_VERSION}")
print(f"  TEMPERATURE    : {TEMPERATURE}")
print(f"  MAX_TOKENS     : {MAX_TOKENS}\n")

results_v1 = run_eval(EVAL_SET_TO_RUN, use_rewrite=False)

print("\n")
results_v2 = run_eval(EVAL_SET_TO_RUN, use_rewrite=True)

avg_time_v1 = round(sum(r["elapsed_sec"] for r in results_v1) / max(len(results_v1), 1), 3)
avg_time_v2 = round(sum(r["elapsed_sec"] for r in results_v2) / max(len(results_v2), 1), 3)

print(f"\n✅ Cell 14 완료 — 평가 실행 완료")
print(f"  V1 평균 응답시간: {avg_time_v1}s")
print(f"  V2 평균 응답시간: {avg_time_v2}s")

🔄 평가 시작 — hf_co_lmstudio_community_exaone_3_5_7_8b_instruct_gguf_latest_v7e | 500개 질문
  PROMPT_VERSION : v7e
  TEMPERATURE    : 0.0
  MAX_TOKENS     : 1000



## cell 15

In [ ]:
# Cell 15 — RAGAS 평가 (선택 실행)
# ※ Quick eval 확인 후 전체 실행 시에만 사용
# ※ USE_QUICK_EVAL=True 상태에서는 생략 가능

RUN_RAGAS = not USE_QUICK_EVAL   # Quick eval 시 RAGAS 생략

if RUN_RAGAS:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.run_config import RunConfig
    from langchain_ollama import ChatOllama
    from langchain_huggingface import HuggingFaceEmbeddings

    ragas_llm = LangchainLLMWrapper(
        ChatOllama(model=RAGAS_LLM_MODEL, base_url="http://localhost:11434", temperature=0)
    )
    ragas_embeddings = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL,
            model_kwargs={"device": DEVICE},
            encode_kwargs={"normalize_embeddings": True}
        )
    )
    ragas_run_config = RunConfig(timeout=1800, max_workers=1, max_retries=2, max_wait=60)

    def run_ragas(results: list[dict], label: str, type_filter: str = None) -> dict:
        filtered = [
            r for r in results
            if not r.get("error") and r.get("contexts")
            and (type_filter is None or r["type"] == type_filter)
        ]
        if not filtered:
            return {"label": label, "count": 0,
                    "faithfulness": None, "answer_relevancy": None, "context_precision": None}

        print(f"\n🔄 RAGAS: {label} | {len(filtered)}개")
        all_scores = {"faithfulness": [], "answer_relevancy": [], "context_precision": []}

        for i, r in enumerate(filtered):
            print(f"  [{i+1}/{len(filtered)}] {r['id']} ...")
            try:
                dataset = Dataset.from_dict({
                    "question": [r["question"]], "answer": [r["parsed_answer"]],
                    "contexts": [r["contexts"]], "ground_truth": [r["ground_truth"]]
                })
                scores = evaluate(dataset,
                                  metrics=[faithfulness, answer_relevancy, context_precision],
                                  llm=ragas_llm, embeddings=ragas_embeddings,
                                  run_config=ragas_run_config)
                scores_df = scores.to_pandas()
                for m in all_scores:
                    val = pd.to_numeric(scores_df[m], errors="coerce").dropna()
                    if len(val) > 0:
                        all_scores[m].append(round(float(val.iloc[0]), 4))
            except Exception as e:
                print(f"    ⚠️ {r['id']} 실패: {e}")

        def safe_mean(lst):
            return round(float(pd.Series(lst).mean()), 4) if lst else None

        result = {
            "label":             label,
            "count":             len(filtered),
            "faithfulness":      safe_mean(all_scores["faithfulness"]),
            "answer_relevancy":  safe_mean(all_scores["answer_relevancy"]),
            "context_precision": safe_mean(all_scores["context_precision"]),
        }
        print(f"  {result}")
        return result

    scores_v1_all   = run_ragas(results_v1, "V1_all")
    scores_v2_all   = run_ragas(results_v2, "V2_all")
    scores_v1_typeC = run_ragas(results_v1, "V1_TypeC", type_filter="C")
    scores_v2_typeC = run_ragas(results_v2, "V2_TypeC", type_filter="C")
    print("✅ Cell 15 완료 — RAGAS 평가 완료")
else:
    scores_v1_all   = scores_v2_all   = None
    scores_v1_typeC = scores_v2_typeC = None
    print("ℹ️ Cell 15 — Quick eval 모드: RAGAS 생략")

ℹ️ Cell 15 — Quick eval 모드: RAGAS 생략


## cell 16

In [ ]:
# Cell 16 — 커스텀 평가
# prom09: wrong_amount_answer_rate + 복합 금액 파서 개선

def normalize_match_text(text: str) -> str:
    text = nfc(str(text))
    text = re.sub(r"\.[A-Za-z0-9]+$", "", text)
    text = re.sub(r"[\s_()\[\]{}·\-.,/「」『』'\"“”‘’]", "", text)
    return text.lower()

def matches_relevant_doc(retrieved: dict, relevant: dict) -> bool:
    ret_file = normalize_match_text(retrieved.get("파일명", ""))
    ret_source = normalize_match_text(retrieved.get("source", ""))
    ret_business = normalize_match_text(retrieved.get("사업명", ""))
    ret_agency = normalize_match_text(retrieved.get("발주기관", ""))

    rel_file = normalize_match_text(relevant.get("파일명", ""))
    rel_stem = normalize_match_text(relevant.get("파일명_stem", ""))
    rel_business = normalize_match_text(relevant.get("사업명", ""))
    rel_agency = normalize_match_text(relevant.get("발주기관", ""))

    if rel_file and ret_file and rel_file == ret_file:
        return True

    if rel_stem and ret_file and (rel_stem in ret_file or ret_file in rel_stem):
        return True

    if rel_file and ret_source and rel_file in ret_source:
        return True

    if rel_business and ret_business and len(rel_business) >= 6:
        if rel_business in ret_business or ret_business in rel_business:
            return True

    if rel_agency and ret_agency and rel_agency == ret_agency and rel_business and ret_business:
        if rel_business[:10] in ret_business or ret_business[:10] in rel_business:
            return True

    return False

def is_refusal_answer(text: str) -> bool:
    return any(kw in str(text) for kw in REFUSAL_KEYWORDS)

def ground_truth_allows_no_amount(ground_truth: str) -> bool:
    gt = str(ground_truth)
    markers = [
        "미기재", "비공개", "판단 불가", "기재되어 있지",
        "정보는 기재", "정보가 전혀", "찾을 수 없습니다",
        "확인되지 않습니다", "명시되어 있지"
    ]
    return any(m in gt for m in markers)

def parse_money_to_won(text: str) -> list[int]:
    text = str(text)
    values = []

    # 복합 표현: 2억 5천만 원, 1억 2,000만 원
    for m in re.finditer(r"(\d[\d,]*(?:\.\d+)?)\s*억\s*(\d[\d,]*(?:\.\d+)?)\s*천?\s*만\s*원?", text):
        try:
            eok = float(m.group(1).replace(",", "")) * 100_000_000
            man = float(m.group(2).replace(",", "")) * 10_000
            values.append(int(round(eok + man)))
        except Exception:
            pass

    for m in re.finditer(r"(\d[\d,]*(?:\.\d+)?)\s*억\s*(\d[\d,]*(?:\.\d+)?)\s*만\s*원?", text):
        try:
            eok = float(m.group(1).replace(",", "")) * 100_000_000
            man = float(m.group(2).replace(",", "")) * 10_000
            values.append(int(round(eok + man)))
        except Exception:
            pass

    patterns = [
        (r"(\d[\d,]*(?:\.\d+)?)\s*억\s*원", 100_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*억원", 100_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*억", 100_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*백만\s*원", 1_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*백만원", 1_000_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*만\s*원", 10_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*만원", 10_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*천\s*원", 1_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*천원", 1_000),
        (r"(\d[\d,]*(?:\.\d+)?)\s*원", 1),
    ]

    for pat, unit in patterns:
        for m in re.finditer(pat, text):
            raw = m.group(1).replace(",", "")
            try:
                values.append(int(round(float(raw) * unit)))
            except Exception:
                pass

    return sorted(set(values))

def answer_has_meaningful_money(answer: str) -> bool:
    values = parse_money_to_won(answer)
    values = [v for v in values if v >= 10_000]
    return len(values) > 0

def money_values_match(gt_values: list[int], target_values: list[int], tolerance_ratio: float = 0.01) -> bool:
    if not gt_values or not target_values:
        return False

    for gt in gt_values:
        for val in target_values:
            tolerance = max(10_000, int(abs(gt) * tolerance_ratio))
            if abs(gt - val) <= tolerance:
                return True

    return False

def calc_refusal_success_rate(results: list[dict]) -> dict:
    d_results = [
        r for r in results
        if r.get("is_refusal_expected") and not r.get("error")
    ]

    if not d_results:
        return {"rate": None, "success": 0, "total": 0, "failed_ids": []}

    success_rows = [
        r for r in d_results
        if is_refusal_answer(r.get("parsed_answer", ""))
    ]

    failed_ids = [
        r["id"] for r in d_results
        if not is_refusal_answer(r.get("parsed_answer", ""))
    ]

    return {
        "rate": round(len(success_rows) / len(d_results), 4),
        "success": len(success_rows),
        "total": len(d_results),
        "failed_ids": failed_ids,
    }

def calc_false_refusal_rate(results: list[dict]) -> dict:
    non_refusal_expected = [
        r for r in results
        if not r.get("is_refusal_expected") and not r.get("error")
    ]

    if not non_refusal_expected:
        return {"rate": None, "count": 0, "total": 0, "ids": []}

    false_refusals = []

    for r in non_refusal_expected:
        parsed = r.get("parsed_answer", "")
        gt = r.get("ground_truth", "")

        if ground_truth_allows_no_amount(gt):
            continue

        if is_refusal_answer(parsed):
            false_refusals.append(r)

    return {
        "rate": round(len(false_refusals) / len(non_refusal_expected), 4),
        "count": len(false_refusals),
        "total": len(non_refusal_expected),
        "ids": [r["id"] for r in false_refusals],
    }

def calc_false_positive_amount_rate(results: list[dict]) -> dict:
    target_rows = [
        r for r in results
        if not r.get("error") and ground_truth_allows_no_amount(r.get("ground_truth", ""))
    ]

    if not target_rows:
        return {"rate": None, "count": 0, "total": 0, "ids": []}

    fp_rows = [
        r for r in target_rows
        if answer_has_meaningful_money(r.get("parsed_answer", ""))
    ]

    return {
        "rate": round(len(fp_rows) / len(target_rows), 4),
        "count": len(fp_rows),
        "total": len(target_rows),
        "ids": [r["id"] for r in fp_rows],
    }

def calc_wrong_amount_answer_rate(results: list[dict]) -> dict:
    target_rows = []

    for r in results:
        if r.get("error"):
            continue

        if ground_truth_allows_no_amount(r.get("ground_truth", "")):
            continue

        gt_money = [v for v in parse_money_to_won(r.get("ground_truth", "")) if v >= 10_000]
        ans_money = [v for v in parse_money_to_won(r.get("parsed_answer", "")) if v >= 10_000]

        if gt_money and ans_money:
            target_rows.append((r, gt_money, ans_money))

    if not target_rows:
        return {"rate": None, "count": 0, "total": 0, "ids": []}

    wrong_rows = []

    for r, gt_money, ans_money in target_rows:
        if not money_values_match(gt_money, ans_money):
            wrong_rows.append(r)

    return {
        "rate": round(len(wrong_rows) / len(target_rows), 4),
        "count": len(wrong_rows),
        "total": len(target_rows),
        "ids": [r["id"] for r in wrong_rows],
    }

def calc_source_missing_rate(results: list[dict]) -> dict:
    valid_rows = [r for r in results if not r.get("error")]

    if not valid_rows:
        return {"rate": None, "count": 0, "total": 0, "ids": []}

    missing = []

    for r in valid_rows:
        src = get_section(r.get("answer", ""), "출처")
        if src in ("", "—", "해당 없음"):
            missing.append(r)

    return {
        "rate": round(len(missing) / len(valid_rows), 4),
        "count": len(missing),
        "total": len(valid_rows),
        "ids": [r["id"] for r in missing],
    }

def calc_one_char_count(results: list[dict]) -> int:
    return sum(
        1 for r in results
        if r.get("parsed_answer", "") in ("—", "") or len(r.get("parsed_answer", "")) <= 1
    )

# ── Hit@k / MRR / nDCG 계산 ──────────────────────────────────────
def calc_doc_retrieval_metrics(results: list[dict], ks: list[int] = [3, 5, 10]) -> dict:
    max_k = max(ks)

    hit_buckets  = {k: [] for k in ks}   # Hit@k
    all_buckets  = {k: [] for k in ks}   # AllHit@k
    cov_buckets  = {k: [] for k in ks}   # Coverage@k
    rr_buckets   = {k: [] for k in ks}   # MRR@k
    ndcg_buckets = {k: [] for k in ks}   # nDCG@k
    count = 0

    for r in results:
        if r.get("error"):
            continue

        relevant_docs = r.get("evidence_docs", []) or []
        if not relevant_docs:
            continue

        retrieved = r.get("retrieved_docs", [])[:max_k]
        count += 1

        # 각 retrieved 문서가 relevant인지 여부 (순서 유지)
        hit_flags = []
        for ret in retrieved:
            is_hit = any(matches_relevant_doc(ret, rel) for rel in relevant_docs)
            hit_flags.append(1 if is_hit else 0)

        # relevant 중 첫 번째 hit rank (MRR용)
        first_hit_rank = None
        for rank, flag in enumerate(hit_flags, start=1):
            if flag:
                first_hit_rank = rank
                break

        num_relevant = len(relevant_docs)

        for k in ks:
            flags_at_k  = hit_flags[:k]
            ret_at_k    = retrieved[:k]

            # Hit@k: relevant 1개 이상 포함
            any_hit = 1 if sum(flags_at_k) > 0 else 0
            hit_buckets[k].append(any_hit)

            # AllHit@k: relevant 전부 포함
            # relevant 각각에 대해 top-k 내 hit 여부 확인
            all_hit_flags = []
            for rel in relevant_docs:
                found = any(matches_relevant_doc(ret_at_k[i], rel) for i in range(len(ret_at_k)))
                all_hit_flags.append(found)
            all_buckets[k].append(1 if all(all_hit_flags) else 0)

            # Coverage@k
            cov = sum(all_hit_flags) / num_relevant
            cov_buckets[k].append(cov)

            # MRR@k
            if first_hit_rank and first_hit_rank <= k:
                rr_buckets[k].append(1.0 / first_hit_rank)
            else:
                rr_buckets[k].append(0.0)

            # nDCG@k (binary relevance)
            # DCG = sum(rel_i / log2(i+1))
            dcg = sum(
                flags_at_k[i] / math.log2(i + 2)
                for i in range(len(flags_at_k))
            )
            # IDCG = ideal: min(num_relevant, k) hits at top positions
            ideal_hits = min(num_relevant, k)
            idcg = sum(
                1.0 / math.log2(i + 2)
                for i in range(ideal_hits)
            )
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            ndcg_buckets[k].append(ndcg)

    if count == 0:
        empty = {
            **{f"hit_at_{k}": None for k in ks},
            **{f"all_hit_at_{k}": None for k in ks},
            **{f"coverage_at_{k}": None for k in ks},
            **{f"mrr_at_{k}": None for k in ks},
            **{f"ndcg_at_{k}": None for k in ks},
            "retrieval_eval_count": 0,
        }
        return empty

    result = {"retrieval_eval_count": count}
    for k in ks:
        n = len(hit_buckets[k])
        result[f"hit_at_{k}"]      = round(sum(hit_buckets[k])  / n, 4)
        result[f"all_hit_at_{k}"]  = round(sum(all_buckets[k])  / n, 4)
        result[f"coverage_at_{k}"] = round(sum(cov_buckets[k])  / n, 4)
        result[f"mrr_at_{k}"]      = round(sum(rr_buckets[k])   / n, 4)
        result[f"ndcg_at_{k}"]     = round(sum(ndcg_buckets[k]) / n, 4)

    return result


def run_custom_eval(results: list[dict], label: str) -> dict:
    refusal            = calc_refusal_success_rate(results)
    false_refusal      = calc_false_refusal_rate(results)
    false_pos_amount   = calc_false_positive_amount_rate(results)
    wrong_amount       = calc_wrong_amount_answer_rate(results)
    source_missing     = calc_source_missing_rate(results)
    retrieval          = calc_doc_retrieval_metrics(results, ks=[3, 5, 10])
    numeric            = calc_numeric_metrics(results)
    one_char           = calc_one_char_count(results)

    result = {
        "label"       : label,
        "total_count" : len(results),
        "error_count" : sum(1 for r in results if r.get("error")),
        "one_char_count": one_char,

        # ── Refusal ──────────────────────────────────────────────
        "refusal_success_rate"  : refusal["rate"],
        "refusal_success"       : refusal["success"],
        "refusal_total"         : refusal["total"],
        "refusal_failed_ids"    : refusal["failed_ids"],

        "false_refusal_rate"    : false_refusal["rate"],
        "false_refusal_count"   : false_refusal["count"],
        "false_refusal_total"   : false_refusal["total"],
        "false_refusal_ids"     : false_refusal["ids"],

        # ── Amount accuracy ──────────────────────────────────────
        "false_positive_amount_rate" : false_pos_amount["rate"],
        "false_positive_amount_count": false_pos_amount["count"],
        "false_positive_amount_total": false_pos_amount["total"],
        "false_positive_amount_ids"  : false_pos_amount["ids"],

        "wrong_amount_answer_rate"   : wrong_amount["rate"],
        "wrong_amount_answer_count"  : wrong_amount["count"],
        "wrong_amount_answer_total"  : wrong_amount["total"],
        "wrong_amount_answer_ids"    : wrong_amount["ids"],

        # ── Source quality ───────────────────────────────────────
        "source_missing_rate"  : source_missing["rate"],
        "source_missing_count" : source_missing["count"],
        "source_missing_total" : source_missing["total"],
        "source_missing_ids"   : source_missing["ids"],

        # ── Retrieval (Hit@k / MRR@k / nDCG@k) ──────────────────
        "retrieval_eval_count" : retrieval["retrieval_eval_count"],

        "hit_at_3"      : retrieval.get("hit_at_3"),
        "hit_at_5"      : retrieval.get("hit_at_5"),
        "hit_at_10"     : retrieval.get("hit_at_10"),

        "all_hit_at_3"  : retrieval.get("all_hit_at_3"),
        "all_hit_at_5"  : retrieval.get("all_hit_at_5"),
        "all_hit_at_10" : retrieval.get("all_hit_at_10"),

        "coverage_at_3" : retrieval.get("coverage_at_3"),
        "coverage_at_5" : retrieval.get("coverage_at_5"),
        "coverage_at_10": retrieval.get("coverage_at_10"),

        "mrr_at_3"      : retrieval.get("mrr_at_3"),
        "mrr_at_5"      : retrieval.get("mrr_at_5"),
        "mrr_at_10"     : retrieval.get("mrr_at_10"),

        "ndcg_at_3"     : retrieval.get("ndcg_at_3"),
        "ndcg_at_5"     : retrieval.get("ndcg_at_5"),
        "ndcg_at_10"    : retrieval.get("ndcg_at_10"),

        # ── Numeric ──────────────────────────────────────────────
        "numeric_eval_count"           : numeric["numeric_eval_count"],
        "context_number_hit_rate"      : numeric["context_number_hit_rate"],
        "answer_number_hit_rate"       : numeric["answer_number_hit_rate"],
        "evidence_card_number_hit_rate": numeric["evidence_card_number_hit_rate"],
        "question_number_use_rate"     : numeric["question_number_use_rate"],
        "numeric_rows"                 : numeric["numeric_rows"],
    }

    # ── 콘솔 출력 ────────────────────────────────────────────────
    print(f"\n📊 [{label}] 커스텀 평가 결과")
    print(f"  총 {result['total_count']}건 | 오류 {result['error_count']}건 | 1자 {result['one_char_count']}건")

    print(f"\n  [Retrieval] 평가 대상: {result['retrieval_eval_count']}건")
    for k in [3, 5, 10]:
        print(f"    Hit@{k}={result[f'hit_at_{k}']} | "
              f"MRR@{k}={result[f'mrr_at_{k}']} | "
              f"nDCG@{k}={result[f'ndcg_at_{k}']} | "
              f"Coverage@{k}={result[f'coverage_at_{k}']}")

    print(f"\n  [Refusal]")
    print(f"    거부 성공률  : {result['refusal_success_rate']} ({result['refusal_success']}/{result['refusal_total']})")
    print(f"    거부 오류율  : {result['false_refusal_rate']} ({result['false_refusal_count']}/{result['false_refusal_total']})")

    print(f"\n  [Amount]")
    print(f"    금액 오답률  : {result['wrong_amount_answer_rate']} ({result['wrong_amount_answer_count']}/{result['wrong_amount_answer_total']})")
    print(f"    금액 오생성률: {result['false_positive_amount_rate']} ({result['false_positive_amount_count']}/{result['false_positive_amount_total']})")

    print(f"\n  [Numeric hit]")
    print(f"    컨텍스트: {result['context_number_hit_rate']} | "
          f"답변: {result['answer_number_hit_rate']} | "
          f"출처누락: {result['source_missing_rate']}")

    return result


custom_v1 = run_custom_eval(results_v1, "V1_custom")
custom_v2 = run_custom_eval(results_v2, "V2_custom")

print("\n✅ Cell 16 완료 — 커스텀 평가 완료")


📊 [V1_custom] 커스텀 평가 결과
  total_count: 15
  error_count: 0
  one_char_count: 0
  refusal_success_rate: 0.5
  refusal_success: 1
  refusal_total: 2
  refusal_failed_ids: ['Q058']
  false_refusal_rate: 0.0769
  false_refusal_count: 1
  false_refusal_total: 13
  false_refusal_ids: ['Q008']
  false_positive_amount_rate: 0.3333
  false_positive_amount_count: 1
  false_positive_amount_total: 3
  false_positive_amount_ids: ['Q179']
  wrong_amount_answer_rate: 0.375
  wrong_amount_answer_count: 3
  wrong_amount_answer_total: 8
  wrong_amount_answer_ids: ['Q008', 'Q107', 'Q328']
  source_missing_rate: 0.0
  source_missing_count: 0
  source_missing_total: 15
  source_missing_ids: []
  doc_any_hit_at_5: 1.0
  doc_all_hit_at_5: 0.8667
  doc_coverage_at_5: 0.9333
  mrr_at_5: 0.8578
  retrieval_eval_count: 15
  numeric_eval_count: 12
  context_number_hit_rate: 0.7
  answer_number_hit_rate: 0.6
  evidence_card_number_hit_rate: 0.625
  question_number_use_rate: 0.8571

📊 [V2_custom] 커스텀 평가 결과
  total

## cell 17

In [ ]:
# Cell 17 — 결과 저장

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass

    return obj

final_results = {
    "experiment_tag": EXPERIMENT_TAG,
    "notebook_version": NOTEBOOK_VERSION,
    "prompt_version": PROMPT_VERSION,
    "llm_model": LLM_MODEL,
    "ragas_llm_model": RAGAS_LLM_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "collection_name": COLLECTION_NAME,
    "data_source": str(CHUNKS_PATH.name),
    "eval_source": str(EVAL_PATH.name),
    "eval_count": len(EVAL_SET_TO_RUN),
    "use_quick_eval": USE_QUICK_EVAL,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "answer_top_k": ANSWER_TOP_K,
    "retrieval_eval_k": RETRIEVAL_EVAL_K,

    # prom09 parameters
    "numeric_chunk_boost": NUMERIC_CHUNK_BOOST,
    "agency_chunk_boost": AGENCY_CHUNK_BOOST,
    "evidence_chunk_boost": EVIDENCE_CHUNK_BOOST,
    "business_overlap_boost": BUSINESS_OVERLAP_BOOST,
    "core_amount_boost": CORE_AMOUNT_BOOST,
    "wrong_business_penalty": WRONG_BUSINESS_PENALTY,
    "source_autofill_top_n": SOURCE_AUTOFILL_TOP_N,
    "use_deterministic_budget_answer": USE_DETERMINISTIC_BUDGET_ANSWER,
    "use_deterministic_multi_amount_calc": USE_DETERMINISTIC_MULTI_AMOUNT_CALC,
    "use_deterministic_duration_answer": USE_DETERMINISTIC_DURATION_ANSWER,
    "deterministic_budget_min_score": DETERMINISTIC_BUDGET_MIN_SCORE,

    "avg_response_sec": {
        "v1": avg_time_v1,
        "v2": avg_time_v2,
    },
    "ragas_scores": {
        "v1_all": scores_v1_all,
        "v2_all": scores_v2_all,
        "v1_typeC": scores_v1_typeC,
        "v2_typeC": scores_v2_typeC,
    },
    "custom_scores": {
        "v1": custom_v1,
        "v2": custom_v2,
    },
    "raw_results": {
        "v1": results_v1,
        "v2": results_v2,
    },
}

safe_results = make_json_safe(final_results)

with open(RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(safe_results, f, ensure_ascii=False, indent=2)

print("✅ Cell 17 완료 — 결과 저장 완료")
print(f"  저장 경로: {RESULT_PATH}")
print(f"  평가 건수: {len(EVAL_SET_TO_RUN)}개")
print(f"  V1 avg  : {avg_time_v1}s")
print(f"  V2 avg  : {avg_time_v2}s")

print("\n  커스텀 지표 요약 (V1 / V2)")
print(f"  1자 답변                  : {custom_v1['one_char_count']} / {custom_v2['one_char_count']}")
print(f"  거부 성공률               : {custom_v1['refusal_success_rate']} / {custom_v2['refusal_success_rate']}")
print(f"  과잉 거부율               : {custom_v1['false_refusal_rate']} / {custom_v2['false_refusal_rate']}")
print(f"  금액 생성 오류율           : {custom_v1['false_positive_amount_rate']} / {custom_v2['false_positive_amount_rate']}")
print(f"  틀린 금액 답변율           : {custom_v1['wrong_amount_answer_rate']} / {custom_v2['wrong_amount_answer_rate']}")
print(f"  출처 누락률               : {custom_v1['source_missing_rate']} / {custom_v2['source_missing_rate']}")
print(f"  문서 Any Hit@5            : {custom_v1['doc_any_hit_at_5']} / {custom_v2['doc_any_hit_at_5']}")
print(f"  문서 All Hit@5            : {custom_v1['doc_all_hit_at_5']} / {custom_v2['doc_all_hit_at_5']}")
print(f"  문서 Coverage@5           : {custom_v1['doc_coverage_at_5']} / {custom_v2['doc_coverage_at_5']}")
print(f"  MRR@5                     : {custom_v1['mrr_at_5']} / {custom_v2['mrr_at_5']}")
print(f"  Context 정답 수치 포함률   : {custom_v1['context_number_hit_rate']} / {custom_v2['context_number_hit_rate']}")
print(f"  Evidence 정답 수치 포함률  : {custom_v1['evidence_card_number_hit_rate']} / {custom_v2['evidence_card_number_hit_rate']}")
print(f"  Answer 정답 수치 포함률    : {custom_v1['answer_number_hit_rate']} / {custom_v2['answer_number_hit_rate']}")

if custom_v1.get("false_refusal_ids") or custom_v2.get("false_refusal_ids"):
    print("\n  과잉 거부 ID")
    print(f"  V1: {custom_v1.get('false_refusal_ids')}")
    print(f"  V2: {custom_v2.get('false_refusal_ids')}")

if custom_v1.get("false_positive_amount_ids") or custom_v2.get("false_positive_amount_ids"):
    print("\n  금액 생성 오류 ID")
    print(f"  V1: {custom_v1.get('false_positive_amount_ids')}")
    print(f"  V2: {custom_v2.get('false_positive_amount_ids')}")

if custom_v1.get("wrong_amount_answer_ids") or custom_v2.get("wrong_amount_answer_ids"):
    print("\n  틀린 금액 답변 ID")
    print(f"  V1: {custom_v1.get('wrong_amount_answer_ids')}")
    print(f"  V2: {custom_v2.get('wrong_amount_answer_ids')}")

✅ Cell 17 완료 — 결과 저장 완료
  저장 경로: /Users/who/Desktop/code_it/project01/team_w2_02/results/eval_results_prom09_hf_co_lmstudio_community_exaone_3_5_7_8b_instruct_gguf_latest_v7e.json
  평가 건수: 15개
  V1 avg  : 21.089s
  V2 avg  : 21.333s

  커스텀 지표 요약 (V1 / V2)
  1자 답변                  : 0 / 0
  거부 성공률               : 0.5 / 0.5
  과잉 거부율               : 0.0769 / 0.0769
  금액 생성 오류율           : 0.3333 / 0.3333
  틀린 금액 답변율           : 0.375 / 0.375
  출처 누락률               : 0.0 / 0.0
  문서 Any Hit@5            : 1.0 / 1.0
  문서 All Hit@5            : 0.8667 / 0.8667
  문서 Coverage@5           : 0.9333 / 0.9333
  MRR@5                     : 0.8578 / 0.8522
  Context 정답 수치 포함률   : 0.7 / 0.7
  Evidence 정답 수치 포함률  : 0.625 / 0.625
  Answer 정답 수치 포함률    : 0.6 / 0.6

  과잉 거부 ID
  V1: ['Q008']
  V2: ['Q008']

  금액 생성 오류 ID
  V1: ['Q179']
  V2: ['Q179']

  틀린 금액 답변 ID
  V1: ['Q008', 'Q107', 'Q328']
  V2: ['Q008', 'Q107', 'Q328']
